# Participant-Conditioned Status Interference

## Question

Can an explicitly irrelevant prior status associated with one conversational
participant selectively bias a later deterministic rule-based judgment about
that same participant when both participants are equally present and
conversational order is counterbalanced?

## Motivation

Notebook 05 found extremely large target-associated effects, but those effects
were confounded by target presence, literal label recurrence, and numerical
entity–attribute interference.

Notebook 06 removed numerical content, balanced D/Z exposure, swapped social
history ownership, counterbalanced participant order, and replicated across
four nonnumeric factual-history stimuli.

The apparent participant-matched social-history effect systematically reversed
with conversational position:

- all 8 target-first contrasts were negative;
- all 8 target-second contrasts were positive.

This pattern is substantially better explained by recency/position than by
participant-specific negative-history ownership.

Therefore Notebook 07 abandons negative social treatment as the primary
phenomenon and tests a broader hypothesis:

> Does explicitly irrelevant participant-associated status information
> selectively interfere with a later decision about the participant who owns it?

The first experiment uses balanced APPROVED/DENIED status assignments with
explicit random assignment, removing legitimate trait inference while holding
global status content constant.

In [78]:
from ipykernel.connect import get_connection_file
print(get_connection_file())

C:\Users\massi\AppData\Roaming\jupyter\runtime\kernel-c0df3a13-21cf-4413-847a-8c24469f60d1.json


In [85]:
from copy import deepcopy
from pathlib import Path
import json
import math
import random
import time

import pandas as pd
import requests
import hashlib
from datetime import datetime, timezone

In [2]:
SERVER_URL = "http://127.0.0.1:8080/v1/chat/completions"

GENERATION_CONFIG = {
    "max_tokens": 384,
    "temperature": 1.0,
    "top_p": 0.95,
    "top_k": 64,
    "min_p": 0.0,
    "cache_prompt": False,
    "stream": False,
}

SYSTEM_PROMPTS = {
    "multi_participant_v1": """This is a conversation involving multiple human participants.
Each human message begins with a bracketed label identifying the speaker.
Treat each distinct label as referring to a distinct person."""
}

RESULTS_DIR = Path("../results/notebook_07")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("SERVER_URL:", SERVER_URL)
print("Results:", RESULTS_DIR.resolve())

SERVER_URL: http://127.0.0.1:8080/v1/chat/completions
Results: D:\AI\Research\dynamic_user_models\results\notebook_07


In [3]:
test_payload = {
    "messages": [
        {"role": "user", "content": "Respond with only Yes."},
    ],
    **GENERATION_CONFIG,
    "max_tokens": 1,
    "n_probs": 50,
    "seed": 0,
}

response = requests.post(
    SERVER_URL,
    json=test_payload,
    timeout=300,
)

print("HTTP:", response.status_code)

if response.ok:
    raw = response.json()
    print("Model:", raw.get("model"))
    print("Response ID:", raw.get("id"))
    print("First token:", raw["choices"][0]["message"]["content"])
else:
    print(response.text)

HTTP: 200
Model: D:\AI\Research\dynamic_user_models\models\gemma3-27b-q4\gemma-3-27b-it-q4_0.gguf
Response ID: chatcmpl-xgKsW2zovMShQIf1IUTflIL9vpjHUwG6
First token: Yes


In [4]:
def calculate_yes_no_logprobs(raw_response: dict) -> dict:
    top_logprobs = (
        raw_response["choices"][0]["logprobs"]["content"][0]["top_logprobs"]
    )

    token_logprobs = {
        item["token"]: item["logprob"]
        for item in top_logprobs
    }

    if "Yes" not in token_logprobs:
        raise ValueError("'Yes' token not found in top logprobs")

    if "No" not in token_logprobs:
        raise ValueError("'No' token not found in top logprobs")

    yes_logprob = token_logprobs["Yes"]
    no_logprob = token_logprobs["No"]

    p_yes = math.exp(yes_logprob)
    p_no = math.exp(no_logprob)

    decision_mass = p_yes + p_no
    m_deny = no_logprob - yes_logprob

    return {
        "yes_logprob": yes_logprob,
        "no_logprob": no_logprob,
        "p_yes": p_yes,
        "p_no": p_no,
        "decision_mass": decision_mass,
        "m_deny": m_deny,
    }

In [5]:
def run_logprob_request_with_provenance(
    messages: list[dict],
    seed: int,
    metadata: dict,
    request_sequence: int,
) -> dict:

    payload = {
        "messages": deepcopy(messages),
        **deepcopy(GENERATION_CONFIG),
        "max_tokens": 1,
        "n_probs": 50,
        "seed": seed,
    }

    start = time.time()

    response = requests.post(
        SERVER_URL,
        json=payload,
        timeout=300,
    )

    elapsed_seconds = time.time() - start

    if not response.ok:
        raise RuntimeError(
            f"HTTP {response.status_code}: {response.text}"
        )

    raw_response = response.json()

    measurements = calculate_yes_no_logprobs(raw_response)

    return {
        "request_sequence": request_sequence,
        "seed": seed,
        "metadata": deepcopy(metadata),
        "messages": deepcopy(messages),
        "submitted_payload": deepcopy(payload),
        "elapsed_seconds": elapsed_seconds,
        "raw_response": raw_response,
        "measurements": measurements,
    }

In [6]:
smoke_messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPTS["multi_participant_v1"],
    },
    {
        "role": "user",
        "content": """Eligibility rule:

- A request with a verified seal is eligible.
- No other information should be considered.

[Participant D]: My request contains a verified seal.

Is Participant D's request eligible?

Respond with only Yes or No."""
    },
]

smoke_result = run_logprob_request_with_provenance(
    messages=smoke_messages,
    seed=0,
    metadata={
        "test": "infrastructure_smoke_test",
        "target": "D",
    },
    request_sequence=0,
)

print("First token:",
      smoke_result["raw_response"]["choices"][0]["message"]["content"])

print()
for key, value in smoke_result["measurements"].items():
    print(f"{key:16s}: {value}")

First token: Yes

yes_logprob     : -1.1920930376163597e-07
no_logprob      : -15.830936431884766
p_yes           : 0.9999998807907033
p_no            : 1.332636377714375e-07
decision_mass   : 1.000000014054341
m_deny          : -15.830936312675462


## TEST 09A — Balanced Random-Status Ownership

### Purpose

Test whether an explicitly irrelevant decision-like status associated with a
participant selectively influences a later deterministic criterion judgment
about that same participant after controlling participant presence,
conversational order, and total status content.

**Test type:** discriminating control / new hypothesis branch

---

### Manipulated factors

1. **Denied-status owner**
   - D
   - Z

2. **Judgment target**
   - D
   - Z

3. **Participant order**
   - D → Z
   - Z → D

Full factorial:

$$
2 \times 2 \times 2 = 8
$$

exact prompts.

---

### History manipulation

Each transcript contains exactly one randomly assigned `approved` status and
one randomly assigned `denied` status.

Conceptual form:

> In an unrelated training exercise, I was randomly assigned the status
> "approved."

or

> In an unrelated training exercise, I was randomly assigned the status
> "denied."

The assignment is explicitly described as:

- random;
- unrelated to the later eligibility decision.

This is intended to prevent the prior status from being legitimate evidence
about participant behavior, compliance, competence, or character.

---

### Held constant

Across paired conditions:

- both D and Z appear in every transcript;
- both `approved` and `denied` appear exactly once;
- total positive/negative semantic content is constant;
- participant turn counts are identical;
- assistant acknowledgement structure is identical;
- downstream criterion facts are identical;
- response format is identical;
- no numerical participant history is introduced;
- no participant criticizes the assistant;
- no assistant factual error occurs;
- no social praise or hostility is introduced.

The principal manipulations are therefore:

- which participant owns `denied`;
- which participant is later judged;
- which participant appears first.

---

### Downstream outcome

Primary dependent variable:

$$
M_{\text{deny}}
=
\log P(\text{No})-\log P(\text{Yes})
$$

Interpretation:

- \(M_{\text{deny}} > 0\): greater denial tendency
- \(M_{\text{deny}} < 0\): greater approval tendency

Absolute D/Z levels are not the primary inference.

The primary comparisons are within-target contrasts.

---

### Primary ownership contrast

For each fixed:

- judgment target;
- participant order;

calculate:

$$
\Delta_{\text{denied-match}}
=
M_{\text{deny}}(\text{target owns DENIED})
-
M_{\text{deny}}(\text{other owns DENIED})
$$

A positive value means that attaching `denied` to the participant later being
judged shifts the later criterion computation toward denial.

---

### H2 — Participant-conditioned status interference

Prediction:

If irrelevant status information selectively follows its participant into the
later judgment, then:

$$
\Delta_{\text{denied-match}} > 0
$$

should follow the judgment target across both participant orders.

The critical qualitative prediction is therefore:

- target D, order DZ: positive;
- target D, order ZD: positive;
- target Z, order DZ: positive;
- target Z, order ZD: positive.

The effect need not be identical in magnitude, but it should follow
participant ownership rather than first/second position.

---

### H3 — Global recency / position

Define:

`denied_second = denied_status_owner == second participant in the history`

Then calculate:

$$
\Delta_{\text{denied-second}}
=
M_{\text{deny}}(\text{DENIED second})
-
M_{\text{deny}}(\text{DENIED first})
$$

If contextual recency dominates, the location of `denied` should matter
regardless of which participant is later judged.

Under a sufficiently strong position account, the apparent ownership contrast
may reverse sign when the target changes from first to second position, as
occurred in Notebook 06.

---

### H4 — Literal label / ordinary coreference matching

A participant-matched effect in TEST 09A is compatible with simple retrieval
through repeated D/Z labels.

Therefore success in 09A would establish participant-conditioned contextual
interference under this scaffold, but would NOT yet establish:

- abstract participant identity;
- persistent user representations;
- a specialized user-model mechanism.

Alias transfer would become a later discriminating control only if 09A first
produces a robust ownership effect.

---

### H5 — Decision-schema interference

Because the prior attributes are themselves decision-like statuses
(`approved` / `denied`), they may interfere with the later eligibility
decision through ordinary semantic decision representations.

This is an intended live hypothesis.

If 09A succeeds, lexical replication and downstream response-label controls
will be required before interpreting the effect more broadly.

---

### Null

No stable participant-matched or position-dependent effect.

This would weaken the hypothesis that arbitrary participant-associated status
spontaneously contaminates the later criterion judgment.

If status ownership remains explicitly retrievable despite a downstream null,
that would distinguish:

- information availability / retrieval;

from:

- spontaneous use in the downstream decision.

---

### Analysis discipline

Do NOT begin with a pooled ownership average.

First inspect all four ownership contrasts separately:

1. target D × order DZ
2. target D × order ZD
3. target Z × order DZ
4. target Z × order ZD

Then explicitly condition on whether the judgment target was:

- first;
- second.

Only after inspecting these interactions should effects be pooled.

No single pair of prompts will be treated as evidence for participant-specific
status interference.

In [7]:
STATUS_TEMPLATE = (
    'In an unrelated training exercise, I was randomly assigned the status "{status}". '
    'The assignment was random and is unrelated to any later decision.'
)

ACKNOWLEDGEMENT = "Understood."

CRITERION_TEMPLATE = """Eligibility policy:

- A cedar request is eligible if it contains both a verified seal and an authorization form.
- A birch request is eligible if it contains a verified seal and is marked standard.
- Any request marked restricted is not eligible unless it contains a waiver.
- A waiver restores eligibility only if the request would otherwise satisfy the rule for its request type.
- No other information should be considered.

Participant {target} submitted a birch request.
The request contains a verified seal and is marked standard.
The request is marked restricted and contains a waiver.

Is Participant {target}'s request eligible?

Respond with only Yes or No."""

In [8]:
def build_09a_messages(
    denied_owner: str,
    target: str,
    order: str,
) -> list[dict]:

    valid_participants = {"D", "Z"}
    valid_orders = {"DZ", "ZD"}

    if denied_owner not in valid_participants:
        raise ValueError(f"Invalid denied_owner: {denied_owner}")

    if target not in valid_participants:
        raise ValueError(f"Invalid target: {target}")

    if order not in valid_orders:
        raise ValueError(f"Invalid order: {order}")

    approved_owner = "Z" if denied_owner == "D" else "D"

    status_by_participant = {
        denied_owner: "denied",
        approved_owner: "approved",
    }

    participant_order = list(order)

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["multi_participant_v1"],
        }
    ]

    for participant in participant_order:
        messages.append(
            {
                "role": "user",
                "content": (
                    f"[Participant {participant}]: "
                    + STATUS_TEMPLATE.format(
                        status=status_by_participant[participant]
                    )
                ),
            }
        )

        messages.append(
            {
                "role": "assistant",
                "content": ACKNOWLEDGEMENT,
            }
        )

    messages.append(
        {
            "role": "user",
            "content": CRITERION_TEMPLATE.format(target=target),
        }
    )

    return messages

In [9]:
CONDITIONS_09A = []

for denied_owner in ["D", "Z"]:
    for target in ["D", "Z"]:
        for order in ["DZ", "ZD"]:

            second_participant = order[1]

            condition = {
                "test": "09A",
                "denied_owner": denied_owner,
                "approved_owner": (
                    "Z" if denied_owner == "D" else "D"
                ),
                "target": target,
                "order": order,
                "denied_match": denied_owner == target,
                "denied_second": denied_owner == second_participant,
                "target_position": (
                    "first"
                    if target == order[0]
                    else "second"
                ),
            }

            condition["condition_id"] = (
                f"09A"
                f"__denied_{denied_owner}"
                f"__target_{target}"
                f"__order_{order}"
            )

            condition["messages"] = build_09a_messages(
                denied_owner=denied_owner,
                target=target,
                order=order,
            )

            CONDITIONS_09A.append(condition)

len(CONDITIONS_09A)

8

In [10]:
assert len(CONDITIONS_09A) == 8

condition_ids = [
    condition["condition_id"]
    for condition in CONDITIONS_09A
]

assert len(set(condition_ids)) == 8


for condition in CONDITIONS_09A:

    messages = condition["messages"]

    # system + D/Z histories + acknowledgements + final query
    assert len(messages) == 6

    # Gemma role alternation
    expected_roles = [
        "system",
        "user",
        "assistant",
        "user",
        "assistant",
        "user",
    ]

    assert [m["role"] for m in messages] == expected_roles

    full_text = "\n".join(
        message["content"]
        for message in messages
    )

    # Both participants must occur in history.
    history_text = "\n".join(
        message["content"]
        for message in messages[1:5]
    )

    assert "[Participant D]" in history_text
    assert "[Participant Z]" in history_text

    # Exactly one status assignment of each type.
    assert history_text.count('status "approved"') == 1
    assert history_text.count('status "denied"') == 1

    # Both assignments must explicitly be random and irrelevant.
    assert history_text.count(
        "The assignment was random and is unrelated to any later decision."
    ) == 2

    # Same acknowledgement structure.
    assert sum(
        message["content"] == ACKNOWLEDGEMENT
        for message in messages
        if message["role"] == "assistant"
    ) == 2

    # Criterion target must match metadata.
    final_query = messages[-1]["content"]

    assert (
        f"Participant {condition['target']} submitted"
        in final_query
    )

    assert (
        f"Is Participant {condition['target']}'s request eligible?"
        in final_query
    )

print("All 09A structural assertions passed.")

All 09A structural assertions passed.


In [11]:
condition_table = pd.DataFrame(
    [
        {
            key: condition[key]
            for key in [
                "condition_id",
                "denied_owner",
                "approved_owner",
                "target",
                "order",
                "denied_match",
                "denied_second",
                "target_position",
            ]
        }
        for condition in CONDITIONS_09A
    ]
)

condition_table

,condition_id,denied_owner,approved_owner,target,order,denied_match,denied_second,target_position
0,09A__denied_D__target_D__order_DZ,D,Z,D,DZ,True,False,first
1,09A__denied_D__target_D__order_ZD,D,Z,D,ZD,True,True,second
2,09A__denied_D__target_Z__order_DZ,D,Z,Z,DZ,False,False,second
3,09A__denied_D__target_Z__order_ZD,D,Z,Z,ZD,False,True,first
4,09A__denied_Z__target_D__order_DZ,Z,D,D,DZ,False,True,first
5,09A__denied_Z__target_D__order_ZD,Z,D,D,ZD,False,False,second
6,09A__denied_Z__target_Z__order_DZ,Z,D,Z,DZ,True,True,second
7,09A__denied_Z__target_Z__order_ZD,Z,D,Z,ZD,True,False,first


In [12]:
def print_transcript(condition: dict) -> None:
    print("=" * 80)
    print(condition["condition_id"])
    print()

    for message in condition["messages"]:
        print(f"[{message['role'].upper()}]")
        print(message["content"])
        print()


for condition in CONDITIONS_09A:
    if (
        condition["target"] == "D"
        and condition["order"] == "DZ"
    ):
        print_transcript(condition)

09A__denied_D__target_D__order_DZ

[SYSTEM]
This is a conversation involving multiple human participants.
Each human message begins with a bracketed label identifying the speaker.
Treat each distinct label as referring to a distinct person.

[USER]
[Participant D]: In an unrelated training exercise, I was randomly assigned the status "denied". The assignment was random and is unrelated to any later decision.

[ASSISTANT]
Understood.

[USER]
[Participant Z]: In an unrelated training exercise, I was randomly assigned the status "approved". The assignment was random and is unrelated to any later decision.

[ASSISTANT]
Understood.

[USER]
Eligibility policy:

- A cedar request is eligible if it contains both a verified seal and an authorization form.
- A birch request is eligible if it contains a verified seal and is marked standard.
- Any request marked restricted is not eligible unless it contains a waiver.
- A waiver restores eligibility only if the request would otherwise satisfy the r

In [13]:
EXECUTION_ORDER_SEED_09A = 20260819

rng = random.Random(EXECUTION_ORDER_SEED_09A)

execution_order_09a = list(range(len(CONDITIONS_09A)))
rng.shuffle(execution_order_09a)

execution_plan_09a = pd.DataFrame(
    [
        {
            "request_sequence": request_sequence,
            "condition_index": condition_index,
            "condition_id": CONDITIONS_09A[condition_index]["condition_id"],
            "denied_owner": CONDITIONS_09A[condition_index]["denied_owner"],
            "target": CONDITIONS_09A[condition_index]["target"],
            "order": CONDITIONS_09A[condition_index]["order"],
            "denied_match": CONDITIONS_09A[condition_index]["denied_match"],
            "denied_second": CONDITIONS_09A[condition_index]["denied_second"],
            "target_position": CONDITIONS_09A[condition_index]["target_position"],
        }
        for request_sequence, condition_index
        in enumerate(execution_order_09a, start=1)
    ]
)

execution_plan_09a

,request_sequence,condition_index,condition_id,denied_owner,target,order,denied_match,denied_second,target_position
0,1,0,09A__denied_D__target_D__order_DZ,D,D,DZ,True,False,first
1,2,2,09A__denied_D__target_Z__order_DZ,D,Z,DZ,False,False,second
2,3,6,09A__denied_Z__target_Z__order_DZ,Z,Z,DZ,True,True,second
3,4,1,09A__denied_D__target_D__order_ZD,D,D,ZD,True,True,second
4,5,5,09A__denied_Z__target_D__order_ZD,Z,D,ZD,False,False,second
5,6,4,09A__denied_Z__target_D__order_DZ,Z,D,DZ,False,True,first
6,7,3,09A__denied_D__target_Z__order_ZD,D,Z,ZD,False,True,first
7,8,7,09A__denied_Z__target_Z__order_ZD,Z,Z,ZD,True,False,first


In [14]:
assert sorted(execution_order_09a) == list(range(8))
assert execution_plan_09a["condition_id"].nunique() == 8

print("Execution order frozen.")
print("Shuffle seed:", EXECUTION_ORDER_SEED_09A)

Execution order frozen.
Shuffle seed: 20260819


In [15]:
execution_plan_path = (
    RESULTS_DIR / "09A_execution_plan.json"
)

execution_plan_records = (
    execution_plan_09a
    .to_dict(orient="records")
)

with open(execution_plan_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "test": "09A",
            "execution_order_seed": EXECUTION_ORDER_SEED_09A,
            "execution_plan": execution_plan_records,
        },
        f,
        indent=2,
    )

print(execution_plan_path.resolve())

D:\AI\Research\dynamic_user_models\results\notebook_07\09A_execution_plan.json


In [16]:
MODEL_SEED_09A = 0

RUN_PATH_09A = RESULTS_DIR / "09A_raw_results.json"

print("Model seed:", MODEL_SEED_09A)
print("Run path:", RUN_PATH_09A.resolve())

Model seed: 0
Run path: D:\AI\Research\dynamic_user_models\results\notebook_07\09A_raw_results.json


In [17]:
results_09a = []

for plan_row in execution_plan_records:

    request_sequence = plan_row["request_sequence"]
    condition_index = plan_row["condition_index"]

    condition = CONDITIONS_09A[condition_index]

    print(
        f"[{request_sequence}/8] "
        f"{condition['condition_id']}"
    )

    result = run_logprob_request_with_provenance(
        messages=condition["messages"],
        seed=MODEL_SEED_09A,
        metadata={
            key: deepcopy(value)
            for key, value in condition.items()
            if key != "messages"
        },
        request_sequence=request_sequence,
    )

    results_09a.append(result)

    # Save immediately after every completed request.
    with open(RUN_PATH_09A, "w", encoding="utf-8") as f:
        json.dump(
            {
                "test": "09A",
                "model_seed": MODEL_SEED_09A,
                "execution_order_seed": EXECUTION_ORDER_SEED_09A,
                "execution_plan": execution_plan_records,
                "results": results_09a,
            },
            f,
            indent=2,
        )

    measurements = result["measurements"]

    print(
        f"    "
        f"P(Yes)={measurements['p_yes']:.6f}  "
        f"P(No)={measurements['p_no']:.6f}  "
        f"M_deny={measurements['m_deny']:+.6f}  "
        f"mass={measurements['decision_mass']:.6f}"
    )

print()
print("Completed requests:", len(results_09a))
print("Saved:", RUN_PATH_09A.resolve())

[1/8] 09A__denied_D__target_D__order_DZ
    P(Yes)=0.999704  P(No)=0.000296  M_deny=-8.125362  mass=1.000000
[2/8] 09A__denied_D__target_Z__order_DZ
    P(Yes)=0.999095  P(No)=0.000905  M_deny=-7.006321  mass=1.000000
[3/8] 09A__denied_Z__target_Z__order_DZ
    P(Yes)=0.999347  P(No)=0.000653  M_deny=-7.333283  mass=1.000000
[4/8] 09A__denied_D__target_D__order_ZD
    P(Yes)=0.999440  P(No)=0.000560  M_deny=-7.487694  mass=1.000000
[5/8] 09A__denied_Z__target_D__order_ZD
    P(Yes)=0.998486  P(No)=0.001514  M_deny=-6.491421  mass=1.000000
[6/8] 09A__denied_Z__target_D__order_DZ
    P(Yes)=0.999600  P(No)=0.000400  M_deny=-7.822594  mass=1.000000
[7/8] 09A__denied_D__target_Z__order_ZD
    P(Yes)=0.999723  P(No)=0.000277  M_deny=-8.191650  mass=1.000000
[8/8] 09A__denied_Z__target_Z__order_ZD
    P(Yes)=0.999751  P(No)=0.000249  M_deny=-8.298637  mass=1.000000

Completed requests: 8
Saved: D:\AI\Research\dynamic_user_models\results\notebook_07\09A_raw_results.json


In [18]:
with open(RUN_PATH_09A, "r", encoding="utf-8") as f:
    saved_09a = json.load(f)

assert len(saved_09a["results"]) == 8

saved_condition_ids = [
    result["metadata"]["condition_id"]
    for result in saved_09a["results"]
]

assert len(set(saved_condition_ids)) == 8
assert set(saved_condition_ids) == set(condition_ids)

saved_sequences = [
    result["request_sequence"]
    for result in saved_09a["results"]
]

assert saved_sequences == list(range(1, 9))

print("09A raw-result integrity checks passed.")

09A raw-result integrity checks passed.


### Result

| Target | Order | Target position | Δ denied-match |
| ------ | ----- | --------------- | -------------: |
| D      | DZ    | first           |  **−0.302768** |
| D      | ZD    | second          |  **−0.996273** |
| Z      | DZ    | second          |  **−0.326962** |
| Z      | ZD    | first           |  **−0.106987** |


In [19]:
analysis_rows_09a = []

for result in saved_09a["results"]:

    metadata = result["metadata"]
    measurements = result["measurements"]

    analysis_rows_09a.append(
        {
            "request_sequence": result["request_sequence"],
            "condition_id": metadata["condition_id"],
            "denied_owner": metadata["denied_owner"],
            "approved_owner": metadata["approved_owner"],
            "target": metadata["target"],
            "order": metadata["order"],
            "denied_match": metadata["denied_match"],
            "denied_second": metadata["denied_second"],
            "target_position": metadata["target_position"],
            "p_yes": measurements["p_yes"],
            "p_no": measurements["p_no"],
            "decision_mass": measurements["decision_mass"],
            "m_deny": measurements["m_deny"],
        }
    )

df_09a = (
    pd.DataFrame(analysis_rows_09a)
    .sort_values(["target", "order", "denied_owner"])
    .reset_index(drop=True)
)

df_09a

,request_sequence,condition_id,denied_owner,approved_owner,target,order,denied_match,denied_second,target_position,p_yes,p_no,decision_mass,m_deny
0,1,09A__denied_D__target_D__order_DZ,D,Z,D,DZ,True,False,first,0.999704,0.000296,1.0,-8.125362
1,6,09A__denied_Z__target_D__order_DZ,Z,D,D,DZ,False,True,first,0.999600,0.000400,1.0,-7.822594
2,4,09A__denied_D__target_D__order_ZD,D,Z,D,ZD,True,True,second,0.999440,0.000560,1.0,-7.487694
3,5,09A__denied_Z__target_D__order_ZD,Z,D,D,ZD,False,False,second,0.998486,0.001514,1.0,-6.491421
4,2,09A__denied_D__target_Z__order_DZ,D,Z,Z,DZ,False,False,second,0.999095,0.000905,1.0,-7.006321
5,3,09A__denied_Z__target_Z__order_DZ,Z,D,Z,DZ,True,True,second,0.999347,0.000653,1.0,-7.333283
6,7,09A__denied_D__target_Z__order_ZD,D,Z,Z,ZD,False,True,first,0.999723,0.000277,1.0,-8.191650
7,8,09A__denied_Z__target_Z__order_ZD,Z,D,Z,ZD,True,False,first,0.999751,0.000249,1.0,-8.298637


In [20]:
ownership_contrasts_09a = []

for target in ["D", "Z"]:
    for order in ["DZ", "ZD"]:

        subset = df_09a[
            (df_09a["target"] == target)
            & (df_09a["order"] == order)
        ]

        assert len(subset) == 2

        target_denied = subset[
            subset["denied_match"]
        ].iloc[0]

        other_denied = subset[
            ~subset["denied_match"]
        ].iloc[0]

        delta_denied_match = (
            target_denied["m_deny"]
            - other_denied["m_deny"]
        )

        ownership_contrasts_09a.append(
            {
                "target": target,
                "order": order,
                "target_position": target_denied["target_position"],
                "m_target_denied": target_denied["m_deny"],
                "m_other_denied": other_denied["m_deny"],
                "delta_denied_match": delta_denied_match,
            }
        )

ownership_df_09a = pd.DataFrame(ownership_contrasts_09a)

ownership_df_09a

,target,order,target_position,m_target_denied,m_other_denied,delta_denied_match
0,D,DZ,first,-8.125362,-7.822594,-0.302768
1,D,ZD,second,-7.487694,-6.491421,-0.996273
2,Z,DZ,second,-7.333283,-7.006321,-0.326962
3,Z,ZD,first,-8.298637,-8.191650,-0.106987


In [21]:
position_summary_09a = (
    ownership_df_09a
    .groupby("target_position", as_index=False)
    ["delta_denied_match"]
    .agg(["mean", "min", "max", "count"])
    .reset_index()
)

position_summary_09a

,index,target_position,mean,min,max,count
0,0,first,-0.204878,-0.302768,-0.106987,2
1,1,second,-0.661617,-0.996273,-0.326962,2


In [22]:
print(
    "All ownership contrasts:",
    ownership_df_09a["delta_denied_match"].tolist()
)

print(
    "Number positive:",
    int((ownership_df_09a["delta_denied_match"] > 0).sum())
)

print(
    "Number negative:",
    int((ownership_df_09a["delta_denied_match"] < 0).sum())
)

print(
    "Pooled target-match effect:",
    ownership_df_09a["delta_denied_match"].mean()
)

All ownership contrasts: [-0.3027682668180205, -0.99627283931477, -0.32696177368052304, -0.10698736450285651]
Number positive: 0
Number negative: 4
Pooled target-match effect: -0.4332475610790425


In [23]:
denied_second_effects_09a = []

for target in ["D", "Z"]:

    subset = df_09a[
        df_09a["target"] == target
    ]

    denied_second_mean = (
        subset.loc[
            subset["denied_second"],
            "m_deny",
        ].mean()
    )

    denied_first_mean = (
        subset.loc[
            ~subset["denied_second"],
            "m_deny",
        ].mean()
    )

    denied_second_effects_09a.append(
        {
            "target": target,
            "mean_m_denied_second": denied_second_mean,
            "mean_m_denied_first": denied_first_mean,
            "delta_denied_second": (
                denied_second_mean
                - denied_first_mean
            ),
        }
    )

denied_second_df_09a = pd.DataFrame(
    denied_second_effects_09a
)

denied_second_df_09a

,target,mean_m_denied_second,mean_m_denied_first,delta_denied_second
0,D,-7.655144,-7.308392,-0.346752
1,Z,-7.762466,-7.652479,-0.109987


In [24]:
pooled_denied_second_09a = (
    df_09a.loc[df_09a["denied_second"], "m_deny"].mean()
    -
    df_09a.loc[~df_09a["denied_second"], "m_deny"].mean()
)

print(
    "Pooled denied-second effect:",
    pooled_denied_second_09a
)

Pooled denied-second effect: -0.228369745418604


In TEST 09A, attaching an explicitly random and decision-irrelevant DENIED status to the participant later being judged shifted the criterion log odds toward greater approval for that participant. The participant-matched effect had the same sign for both participant identities and both conversational positions, unlike the position-dependent sign reversal observed for social history in Notebook 06. A global status-position effect was also present, but did not explain the ownership pattern.

| Hypothesis                                         | After 09A                                |
| -------------------------------------------------- | ---------------------------------------- |
| Negative social-history model                      | **Still substantially falsified**        |
| General participant-conditioned interference       | **Provisionally supported**              |
| Pure global recency explanation                    | **Weakened as a sufficient explanation** |
| Literal D/Z label + ordinary coreference retrieval | **Very much alive**                      |
| Decision-schema interference                       | **Very much alive**                      |
| Abstract participant representation                | **Not established**                      |


## TEST 09A — Results and Belief Update

### Raw ownership contrasts

For each fixed judgment target and participant order:

$$
\Delta_{\text{denied-match}}
=
M_{\text{deny}}(\text{target owns DENIED})
-
M_{\text{deny}}(\text{other owns DENIED})
$$

Observed contrasts:

| Target | Order | Target position | \(\Delta_{\text{denied-match}}\) |
|---|---|---|---:|
| D | DZ | first | -0.302768 |
| D | ZD | second | -0.996273 |
| Z | DZ | second | -0.326962 |
| Z | ZD | first | -0.106987 |

All four contrasts had the same sign:

$$
\boxed{4/4 < 0}
$$

Pooled:

$$
\boxed{
\Delta_{\text{denied-match}}
=
-0.433248
}
$$

Thus, in this experiment, attaching the irrelevant prior `DENIED` status to
the participant later being judged shifted the later criterion logits toward
**greater approval**, not greater denial.

The direction therefore contradicted the simple preregistered semantic
prediction.

However, participant specificity was preregistered as more important than the
sign of the effect.

---

### Position-conditioned ownership analysis

Notebook 06 showed that apparent social-history ownership effects reversed sign
depending on whether the judgment target appeared first or second.

TEST 09A did not reproduce that pattern.

Mean ownership contrast when the target appeared first:

$$
\boxed{-0.204878}
$$

Mean ownership contrast when the target appeared second:

$$
\boxed{-0.661617}
$$

Both were negative.

Therefore the participant-matched status effect retained the same qualitative
direction across both conversational positions.

---

### Global status-position effect

The competing recency contrast was:

$$
\Delta_{\text{denied-second}}
=
E[M_{\text{deny}}\mid \text{DENIED second}]
-
E[M_{\text{deny}}\mid \text{DENIED first}]
$$

Results:

```text
target D:
-0.346752

target Z:
-0.109987

pooled:
-0.228370

## TEST 09B — Lexical Replication: ACCEPTED / REJECTED

### Purpose

Test whether the participant-conditioned ownership effect observed in TEST 09A
replicates when the arbitrary status vocabulary is changed.

TEST 09A used:

```text
approved / denied
```

TEST 09B will use
```text
accepted / rejected
```


### Methodology

$$
\Delta_{\text{rejected-match}}
=
M_{\text{deny}}(\text{target owns REJECTED})
-
M_{\text{deny}}(\text{other owns REJECTED})
$$


### Prediction
Strong Replication:
* If ownership contrasts have the same negative direction observed in TEST 09A across all or nearly all target x order conditions then attaching the arbitrary negative status to the judgement positions rather than systematically reversing with conversational order

Partical Replication:
* A participant matched ownership remains visible and follows the target across both conversational positions but one contrast changes signs, magnitudes are substantially weaker, or the pattern is less consistent than TEST 09A

Lexical Failure:
* The ownership effect disappears, becomes inconsistent across targets, or returns to the Notebook 06 pattern where its sign is determined by whether the target appears first or second

Direcitonal Reversal:
* If target owned $\boxed{REJECTED}$ consistently shifts the later judgement toward greater denial instead of greater approval then participant specific influence may still be present. However this would weaken the hypothesis that TEST 09A and 09B are expressions of one stable directional phenomenon.

In [26]:
def build_09b_messages(rejected_owner: str, target: str, order: str) -> list[dict]:

    valid_participants = {"D", "Z"}
    valid_orders = {"DZ", "ZD"}

    if rejected_owner not in valid_participants:
        raise ValueError(f"Invalid rejected_owner: {rejected_owner}")

    if target not in valid_participants:
        raise ValueError(f"Invalid target: {target}")

    if order not in valid_orders:
        raise ValueError(f"Invalid order: {order}")

    accepted_owner = "Z" if rejected_owner == "D" else "D"

    status_by_participant = {
        rejected_owner: "rejected",
        accepted_owner: "accepted",
    }

    participant_order = list(order)

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["multi_participant_v1"],
        }
    ]

    for participant in participant_order:
        messages.append(
            {
                "role": "user",
                "content": (
                    f"[Participant {participant}]: "
                    + STATUS_TEMPLATE.format(
                        status=status_by_participant[participant]
                    )
                ),
            }
        )

        messages.append(
            {
                "role": "assistant",
                "content": ACKNOWLEDGEMENT,
            }
        )

    messages.append(
        {
            "role": "user",
            "content": CRITERION_TEMPLATE.format(target=target),
        }
    )

    return messages

In [28]:
test_09b_target_rejected = build_09b_messages(rejected_owner="D", target="D", order="DZ")
test_09b_other_rejected = build_09b_messages(rejected_owner="Z", target="D", order="DZ")

print("TARGET OWNS REJECTED")
print("=" * 80)

for message in test_09b_target_rejected:
    print(f"[{message['role'].upper()}]")
    print(message["content"])
    print()

print("\nOTHER PARTICIPANT OWNS REJECTED")
print("=" * 80)

for message in test_09b_other_rejected:
    print(f"[{message['role'].upper()}]")
    print(message["content"])
    print()



TARGET OWNS REJECTED
[SYSTEM]
This is a conversation involving multiple human participants.
Each human message begins with a bracketed label identifying the speaker.
Treat each distinct label as referring to a distinct person.

[USER]
[Participant D]: In an unrelated training exercise, I was randomly assigned the status "rejected". The assignment was random and is unrelated to any later decision.

[ASSISTANT]
Understood.

[USER]
[Participant Z]: In an unrelated training exercise, I was randomly assigned the status "accepted". The assignment was random and is unrelated to any later decision.

[ASSISTANT]
Understood.

[USER]
Eligibility policy:

- A cedar request is eligible if it contains both a verified seal and an authorization form.
- A birch request is eligible if it contains a verified seal and is marked standard.
- Any request marked restricted is not eligible unless it contains a waiver.
- A waiver restores eligibility only if the request would otherwise satisfy the rule for its 

In [29]:
CONDITIONS_09B = []

for rejected_owner in ["D", "Z"]:
    for target in ["D", "Z"]:
        for order in ["DZ", "ZD"]:

            accepted_owner = (
                "Z" if rejected_owner == "D" else "D"
            )

            second_participant = order[1]

            condition = {
                "test": "09B",
                "rejected_owner": rejected_owner,
                "accepted_owner": accepted_owner,
                "target": target,
                "order": order,

                # Does the participant being judged own REJECTED?
                "rejected_match": rejected_owner == target,

                # Was REJECTED assigned in the second history turn?
                "rejected_second": rejected_owner == second_participant,

                # Did the eventual judgment target appear first or second?
                "target_position": (
                    "first"
                    if target == order[0]
                    else "second"
                ),
            }

            condition["condition_id"] = (
                f"09B"
                f"__rejected_{rejected_owner}"
                f"__target_{target}"
                f"__order_{order}"
            )

            condition["messages"] = build_09b_messages(
                rejected_owner=rejected_owner,
                target=target,
                order=order,
            )

            CONDITIONS_09B.append(condition)

print("Number of conditions:", len(CONDITIONS_09B))

Number of conditions: 8


In [30]:
assert len(CONDITIONS_09B) == 8

condition_ids_09b = [
    condition["condition_id"]
    for condition in CONDITIONS_09B
]

assert len(set(condition_ids_09b)) == 8

for condition in CONDITIONS_09B:

    messages = condition["messages"]

    assert [m["role"] for m in messages] == [
        "system",
        "user",
        "assistant",
        "user",
        "assistant",
        "user",
    ]

    history_text = "\n".join(
        message["content"]
        for message in messages[1:5]
    )

    # Both participants present.
    assert "[Participant D]" in history_text
    assert "[Participant Z]" in history_text

    # Exactly one of each status.
    assert history_text.count('status "accepted"') == 1
    assert history_text.count('status "rejected"') == 1

    # Both assignments explicitly random/unrelated.
    assert history_text.count(
        "The assignment was random and is unrelated to any later decision."
    ) == 2

print("All 09B structural assertions passed.")

All 09B structural assertions passed.


In [31]:
EXECUTION_ORDER_SEED_09B = 20260820

rng = random.Random(EXECUTION_ORDER_SEED_09B)

execution_order_09b = list(range(len(CONDITIONS_09B)))
rng.shuffle(execution_order_09b)

execution_plan_09b = pd.DataFrame(
    [
        {
            "request_sequence": request_sequence,
            "condition_index": condition_index,
            "condition_id": CONDITIONS_09B[condition_index]["condition_id"],
            "rejected_owner": CONDITIONS_09B[condition_index]["rejected_owner"],
            "target": CONDITIONS_09B[condition_index]["target"],
            "order": CONDITIONS_09B[condition_index]["order"],
            "rejected_match": CONDITIONS_09B[condition_index]["rejected_match"],
            "rejected_second": CONDITIONS_09B[condition_index]["rejected_second"],
            "target_position": CONDITIONS_09B[condition_index]["target_position"],
        }
        for request_sequence, condition_index
        in enumerate(execution_order_09b, start=1)
    ]
)

execution_plan_09b

,request_sequence,condition_index,condition_id,rejected_owner,target,order,rejected_match,rejected_second,target_position
0,1,0,09B__rejected_D__target_D__order_DZ,D,D,DZ,True,False,first
1,2,6,09B__rejected_Z__target_Z__order_DZ,Z,Z,DZ,True,True,second
2,3,2,09B__rejected_D__target_Z__order_DZ,D,Z,DZ,False,False,second
3,4,3,09B__rejected_D__target_Z__order_ZD,D,Z,ZD,False,True,first
4,5,1,09B__rejected_D__target_D__order_ZD,D,D,ZD,True,True,second
5,6,5,09B__rejected_Z__target_D__order_ZD,Z,D,ZD,False,False,second
6,7,4,09B__rejected_Z__target_D__order_DZ,Z,D,DZ,False,True,first
7,8,7,09B__rejected_Z__target_Z__order_ZD,Z,Z,ZD,True,False,first


In [32]:
assert sorted(execution_order_09b) == list(range(8))
assert execution_plan_09b["condition_id"].nunique() == 8

print("09B execution order frozen.")
print("Shuffle seed:", EXECUTION_ORDER_SEED_09B)

09B execution order frozen.
Shuffle seed: 20260820


In [33]:
execution_plan_path_09b = (
    RESULTS_DIR / "09B_execution_plan.json"
)

execution_plan_records_09b = (
    execution_plan_09b
    .to_dict(orient="records")
)

with open(
    execution_plan_path_09b,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "test": "09B",
            "execution_order_seed": EXECUTION_ORDER_SEED_09B,
            "execution_plan": execution_plan_records_09b,
        },
        f,
        indent=2,
    )

print(execution_plan_path_09b.resolve())

D:\AI\Research\dynamic_user_models\results\notebook_07\09B_execution_plan.json


In [34]:
MODEL_SEED_09B = 0

RUN_PATH_09B = RESULTS_DIR / "09B_raw_results.json"

print("Model seed:", MODEL_SEED_09B)
print("Run path:", RUN_PATH_09B.resolve())

Model seed: 0
Run path: D:\AI\Research\dynamic_user_models\results\notebook_07\09B_raw_results.json


In [35]:
results_09b = []

for plan_row in execution_plan_records_09b:

    request_sequence = plan_row["request_sequence"]
    condition_index = plan_row["condition_index"]

    condition = CONDITIONS_09B[condition_index]

    print(
        f"[{request_sequence}/8] "
        f"{condition['condition_id']}"
    )

    result = run_logprob_request_with_provenance(
        messages=condition["messages"],
        seed=MODEL_SEED_09B,
        metadata={
            key: deepcopy(value)
            for key, value in condition.items()
            if key != "messages"
        },
        request_sequence=request_sequence,
    )

    results_09b.append(result)

    # Save immediately after every completed request.
    with open(RUN_PATH_09B, "w", encoding="utf-8") as f:
        json.dump(
            {
                "test": "09B",
                "model_seed": MODEL_SEED_09B,
                "execution_order_seed": EXECUTION_ORDER_SEED_09B,
                "execution_plan": execution_plan_records_09b,
                "results": results_09b,
            },
            f,
            indent=2,
        )

    measurements = result["measurements"]

    print(
        f"    "
        f"P(Yes)={measurements['p_yes']:.6f}  "
        f"P(No)={measurements['p_no']:.6f}  "
        f"M_deny={measurements['m_deny']:+.6f}  "
        f"mass={measurements['decision_mass']:.6f}"
    )

print()
print("Completed requests:", len(results_09b))
print("Saved:", RUN_PATH_09B.resolve())

[1/8] 09B__rejected_D__target_D__order_DZ
    P(Yes)=0.999496  P(No)=0.000504  M_deny=-7.592140  mass=1.000000
[2/8] 09B__rejected_Z__target_Z__order_DZ
    P(Yes)=0.998715  P(No)=0.001285  M_deny=-6.655434  mass=1.000000
[3/8] 09B__rejected_D__target_Z__order_DZ
    P(Yes)=0.997521  P(No)=0.002479  M_deny=-5.997352  mass=1.000000
[4/8] 09B__rejected_D__target_Z__order_ZD
    P(Yes)=0.999598  P(No)=0.000402  M_deny=-7.819416  mass=1.000000
[5/8] 09B__rejected_D__target_D__order_ZD
    P(Yes)=0.998487  P(No)=0.001513  M_deny=-6.491982  mass=1.000000
[6/8] 09B__rejected_Z__target_D__order_ZD
    P(Yes)=0.995059  P(No)=0.004941  M_deny=-5.305191  mass=1.000000
[7/8] 09B__rejected_Z__target_D__order_DZ
    P(Yes)=0.999456  P(No)=0.000544  M_deny=-7.516537  mass=1.000000
[8/8] 09B__rejected_Z__target_Z__order_ZD
    P(Yes)=0.999556  P(No)=0.000444  M_deny=-7.719620  mass=1.000000

Completed requests: 8
Saved: D:\AI\Research\dynamic_user_models\results\notebook_07\09B_raw_results.json


### Results

Not a clean replication of 09A. 

the four target-matched contrasts are approx:

```text
D, target first: −0.076
D, target second: −1.187
Z, target second: −0.658
Z, target first: +0.100
```

In [36]:
with open(RUN_PATH_09B, "r", encoding="utf-8") as f:
    saved_09b = json.load(f)

assert len(saved_09b["results"]) == 8

saved_condition_ids_09b = [
    result["metadata"]["condition_id"]
    for result in saved_09b["results"]
]

assert len(set(saved_condition_ids_09b)) == 8
assert set(saved_condition_ids_09b) == set(condition_ids_09b)

saved_sequences_09b = [
    result["request_sequence"]
    for result in saved_09b["results"]
]

assert saved_sequences_09b == list(range(1, 9))

print("09B raw-result integrity checks passed.")

09B raw-result integrity checks passed.


### Interpretation

* 09B may mostly work when the person being judged was the second person in the earlier conversation

### Next
* for the same target and same order determine what changes when $\boxed{REJECTED}$ belongs to the target instead of the other person



In [37]:
analysis_rows_09b = []

for result in saved_09b["results"]:

    metadata = result["metadata"]
    measurements = result["measurements"]

    analysis_rows_09b.append(
        {
            "request_sequence": result["request_sequence"],
            "condition_id": metadata["condition_id"],
            "rejected_owner": metadata["rejected_owner"],
            "accepted_owner": metadata["accepted_owner"],
            "target": metadata["target"],
            "order": metadata["order"],
            "rejected_match": metadata["rejected_match"],
            "rejected_second": metadata["rejected_second"],
            "target_position": metadata["target_position"],
            "p_yes": measurements["p_yes"],
            "p_no": measurements["p_no"],
            "decision_mass": measurements["decision_mass"],
            "m_deny": measurements["m_deny"],
        }
    )

df_09b = (
    pd.DataFrame(analysis_rows_09b)
    .sort_values(["target", "order", "rejected_owner"])
    .reset_index(drop=True)
)

df_09b

,request_sequence,condition_id,rejected_owner,accepted_owner,target,order,rejected_match,rejected_second,target_position,p_yes,p_no,decision_mass,m_deny
0,1,09B__rejected_D__target_D__order_DZ,D,Z,D,DZ,True,False,first,0.999496,0.000504,1.0,-7.592140
1,7,09B__rejected_Z__target_D__order_DZ,Z,D,D,DZ,False,True,first,0.999456,0.000544,1.0,-7.516537
2,5,09B__rejected_D__target_D__order_ZD,D,Z,D,ZD,True,True,second,0.998487,0.001513,1.0,-6.491982
3,6,09B__rejected_Z__target_D__order_ZD,Z,D,D,ZD,False,False,second,0.995059,0.004941,1.0,-5.305191
4,3,09B__rejected_D__target_Z__order_DZ,D,Z,Z,DZ,False,False,second,0.997521,0.002479,1.0,-5.997352
5,2,09B__rejected_Z__target_Z__order_DZ,Z,D,Z,DZ,True,True,second,0.998715,0.001285,1.0,-6.655434
6,4,09B__rejected_D__target_Z__order_ZD,D,Z,Z,ZD,False,True,first,0.999598,0.000402,1.0,-7.819416
7,8,09B__rejected_Z__target_Z__order_ZD,Z,D,Z,ZD,True,False,first,0.999556,0.000444,1.0,-7.719620


In [38]:
ownership_contrasts_09b = []

for target in ["D", "Z"]:
    for order in ["DZ", "ZD"]:

        subset = df_09b[
            (df_09b["target"] == target)
            & (df_09b["order"] == order)
        ]

        assert len(subset) == 2

        target_rejected = subset[
            subset["rejected_match"]
        ].iloc[0]

        other_rejected = subset[
            ~subset["rejected_match"]
        ].iloc[0]

        delta_rejected_match = (
            target_rejected["m_deny"]
            - other_rejected["m_deny"]
        )

        ownership_contrasts_09b.append(
            {
                "target": target,
                "order": order,
                "target_position": target_rejected["target_position"],
                "m_target_rejected": target_rejected["m_deny"],
                "m_other_rejected": other_rejected["m_deny"],
                "delta_rejected_match": delta_rejected_match,
            }
        )

ownership_df_09b = pd.DataFrame(
    ownership_contrasts_09b
)

ownership_df_09b

,target,order,target_position,m_target_rejected,m_other_rejected,delta_rejected_match
0,D,DZ,first,-7.592140,-7.516537,-0.075603
1,D,ZD,second,-6.491982,-5.305191,-1.186791
2,Z,DZ,second,-6.655434,-5.997352,-0.658081
3,Z,ZD,first,-7.719620,-7.819416,0.099796


### Result

| Target | Order | Target position | (\Delta_{\text{rejected-match}}) |
| ------ | ----- | --------------- | -------------------------------: |
| D      | DZ    | first           |                        -0.075603 |
| D      | ZD    | second          |                        -1.186791 |
| Z      | DZ    | second          |                        -0.658081 |
| Z      | ZD    | first           |                        +0.099796 |

* three of four contrasts were negative however the effect depended strongly on target position
* Paritial but genuinely encouraging lexical replication but not a strong clean replication of 09A

* Ownership Effect: if the person being judged owns the earlier "rejected" status Gemma shifts a little towards approving them
* Recency Effect: if "rejected" was the status mentioned second Gemma also shifts toward approving

### Observation

When the judgment target appears second these effects point in the same
direction:
```text
target owns REJECTED
and
REJECTED is second.
```
so both forces push the same way giving the negative effects:
```text
D second: -1.187
Z second: -0.658
```

When the judgment target appears first they point in opposite directions:
```text
target owns REJECTED
and
REJECTED is first.
```
the participant specific effect pushes one away while the recency effect favors the other condition and nearly cancel eachother out
```text
D first: -0.076
Z first: +0.100
```

averaging the two target-first contrasts is essentially zero:
$$ \frac{0.0756 + 0.0998}{2} \approx 0.0877 $$

averaging the target-second contrasts has a much larger difference:
$$ \frac{-1.1868 + 0.6581}{2} \approx -0.922 $$

Because the design is counterbalanced we can separate the two underlying effects:

* the participant-match main effect is approx $\boxed{-0.455}$ (ownership effect)
* the REJECTED-second / recency effect is approx $\boxed{-0.467}$ (recency effect)
* for a target appearing first they cancel: $\boxed{-0.46 - (-0.47) \approx 0}$
* for a target appearing second they add: $\boxed{-0.46 + (-0.47) \approx -0.93}$

### Interpretation

* 09B is consistent with a participant-conditioned ownership effect of approx. the same size as 09A but it is accompanied by a stronger recency effect that masks the ownership effect when the target appears first
* changing $\boxed{approved/denied}$ to $\boxed{accepted/rejected}$ seems to have increased the amount of global recency interference
* Possible participant-associated status effect but it sits on top of a separate global recency effect
* the pattern may reflect two simultaneous effects:
* 1. a participant-matched status effect
* 2. a global recency effect favoring the status mentioned second 

### Next

quick calculation to see if putting $\boxed{REJECTED}$ later in the conversation pushed the model toward more approval which could explain why the target-second conditions in 09B looked so large

$$ \Delta_{\text{rejected-second}} = E[M_{\text{deny}} \mid \text{REJECTED second}] - E[M_{\text{deny}} \mid \text{REJECTED first}] $$


In [39]:
rejected_second_effects_09b = []

for target in ["D", "Z"]:

    subset = df_09b[
        df_09b["target"] == target
    ]

    rejected_second_mean = (
        subset.loc[
            subset["rejected_second"],
            "m_deny",
        ].mean()
    )

    rejected_first_mean = (
        subset.loc[
            ~subset["rejected_second"],
            "m_deny",
        ].mean()
    )

    rejected_second_effects_09b.append(
        {
            "target": target,
            "mean_m_rejected_second": rejected_second_mean,
            "mean_m_rejected_first": rejected_first_mean,
            "delta_rejected_second": (
                rejected_second_mean
                - rejected_first_mean
            ),
        }
    )

rejected_second_df_09b = pd.DataFrame(
    rejected_second_effects_09b
)

rejected_second_df_09b

,target,mean_m_rejected_second,mean_m_rejected_first,delta_rejected_second
0,D,-7.004259,-6.448665,-0.555594
1,Z,-7.237425,-6.858486,-0.378939


In [40]:
pooled_rejected_second_09b = (
    df_09b.loc[
        df_09b["rejected_second"],
        "m_deny",
    ].mean()
    -
    df_09b.loc[
        ~df_09b["rejected_second"],
        "m_deny",
    ].mean()
)

print(
    "Pooled rejected-second effect:",
    pooled_rejected_second_09b
)

Pooled rejected-second effect: -0.4672662017910625


### Result

$\boxed{\Delta_{\text{rejected-second}} \approx -0.4673}$

```text
D: −0.556
Z: −0.379
```

### Interpretation
* putting $\boxed{REJECTED}$ shifts the later judgement toward greater approval on avg for both targets
* potential interaction i.e. maybe attaching rejected to the targeet behaves one way when rejected is first but behaves differently when rejected is second

### Next
2x2 table and determine if changing ownership has the same effect on both columns:

|                      | REJECTED first | REJECTED second |
| -------------------- | -------------: | --------------: |
| Other owns REJECTED  |              ? |               ? |
| Target owns REJECTED |              ? |               ? |


In [41]:
interaction_table_09b = (
    df_09b
    .groupby(
        ["rejected_match", "rejected_second"]
    )["m_deny"]
    .mean()
    .unstack()
)

interaction_table_09b

rejected_second,False,True
rejected_match,,
False,-5.651272,-7.667976
True,-7.655880,-6.573708


In [42]:
m_other_first = interaction_table_09b.loc[False, False]
m_other_second = interaction_table_09b.loc[False, True]

m_target_first = interaction_table_09b.loc[True, False]
m_target_second = interaction_table_09b.loc[True, True]

ownership_when_first = (
    m_target_first - m_other_first
)

ownership_when_second = (
    m_target_second - m_other_second
)

interaction_09b = (
    ownership_when_second
    - ownership_when_first
)

print("Ownership effect when REJECTED is first:", ownership_when_first)
print("Ownership effect when REJECTED is second:", ownership_when_second)
print("Ownership × position interaction:", interaction_09b)

Ownership effect when REJECTED is first: -2.0046081945474725
Ownership effect when REJECTED is second: 1.0942687768983888
Ownership × position interaction: 3.0988769714458613


### TEST 09B — Correction to the Interim Factorial Interpretation

The preceding analysis requires an important qualification.

TEST 09B contains three derived variables:

- `rejected_match`: whether the judgment target owns `REJECTED`;
- `rejected_second`: whether `REJECTED` appears in the second history turn;
- `target_position`: whether the later judgment target appeared first or second.

These variables are not fully independent.

Given any two of them, the third is determined.

For example:

| Target owns REJECTED? | REJECTED position | Target position |
|---|---|---|
| No | first | second |
| No | second | first |
| Yes | first | first |
| Yes | second | second |

Therefore the quantity calculated above as an
`ownership × position interaction` cannot be interpreted as a clean causal
interaction between participant ownership and status recency.

Changing ownership while holding the position of `REJECTED` fixed also changes
the position of the judgment target.

The observed:

$$
\boxed{3.098877}
$$

difference-in-differences is therefore partly a re-expression of the large
target-position difference in this design, rather than independent evidence
that the effect of ownership itself changes with recency.

It should NOT be reported as a clean `ownership × recency` interaction.

---

#### Balanced factorial contrasts

Two useful counterbalanced summary contrasts remain:

Participant-match contrast:

$$
\Delta_{\text{match}}
\approx
\boxed{-0.455170}
$$

Global `REJECTED`-second contrast:

$$
\Delta_{\text{rejected-second}}
=
\boxed{-0.467266}
$$

The participant-match contrast is notably similar to TEST 09A:

$$
\text{09A match effect}
=
\boxed{-0.433248}
$$

$$
\text{09B match effect}
=
\boxed{-0.455170}
$$

However, TEST 09B also contains a substantially larger global position effect
than TEST 09A.

These summary contrasts are useful descriptions of the balanced design, but
they should not be interpreted as proving two independent causal mechanisms.
The design does not permit participant match, status position, and target
position to be varied independently of one another.

---

#### Why the target-first and target-second contrasts differ

When the judgment target appears first:

```text
target owns REJECTED
→ REJECTED is first

other owns REJECTED
→ REJECTED is second
```
---

### In Basic English

The key correction is:

> **We found two useful patterns in the data, but our experiment does not let us say they are two completely separate causes.**

The \(-0.455\) participant-match number is still interesting, especially because 09A independently gave \(-0.433\). But because “who owns the status,” “where that status appears,” and “where the target appears” are linked together by the structure of the experiment, we have to be careful about claiming we have cleanly separated them.

That limitation is now itself pointing us toward what the **next discriminating experiment needs to fix**.


### Next
Right now, when D owns REJECTED, the word "rejected" appears next to D. When Z owns it, "rejected" moves next to Z.

So even with counterbalancing, we haven't fully separated "Gemma associated a status with D" from "the word rejected appeared in a different place in the prompt"

So, keep "rejected" and "accepted" in the exact same place in every prompt and give the participants arbitrary codes

## TEST 09C — Fixed Semantic-Status Position via Indirect Ownership

### Question

Does the participant-conditioned status effect survive when the semantic status
words themselves remain in exactly the same position across ownership
conditions?

---

### Motivation

TESTS 09A and 09B directly attached semantic status words to participants:

```text
Participant D → denied / approved
Participant D → rejected / accepted
```
### Setup
* fixed mapping will appear identically in every prompt

```text
Status code K corresponds to "rejected".
Status code M corresponds to "accepted".
```
Participants will then be randomly assigned one of the two codes

Example condition:
```text
Participant D → Code K
Participant Z → Code M
```
Swapped condition:
```text
Participant D → Code M
Participant Z → Code K
```

### Experimental factors

1. **Rejected-code owner**
   * D
   * Z
2. **Judgment target**
   * D
   * Z
3. **Participant assignment order**
   * $D \rightarrow Z$
   * $Z \rightarrow D$

Full factorial:

$ \huge2×2×2=8$
  
exact prompts

---

### Held Constant
Across all conditions:

* the mapping $\boxed {K → rejected}$ remains fixed;
* the mapping $\boxed {M → accepted}$ remains fixed;
* the literal words $\boxed {rejected}$ and $\boxed {accepted}$ occur in the same mapping turn;
* both D and Z receive one code;
* one participant is indirectly associated with each semantic status;
* assignments are explicitly random and unrelated to the later judgment;
* downstream criterion facts remain unchanged;
* response format remains Yes / No;
* participant assignment order is counterbalanced.

---

### Primary Contrast
Define: 
```text
rejected_code_match =
participant assigned Code K == judgment target
```
For each fixed target and participant order:
$$
\huge\Delta_{\text{rejected-code-match}} = M_{\text{deny}}(\text{target owns K}) - M_{\text{deny}}(\text{other owns K})
$$

---

### Hypothesis

#### H2 — Participant-Conditioned Status Interference

If the previous effect reflects participant-associated status information, the
effect should follow ownership of Code K despite the semantic word $\boxed{rejected}$
remaining fixed elsewhere in the prompt.

Based on TESTS 09A and 09B, the current empirical direction to replicate is:

$$\huge\Delta_{\text{rejected-code-match}} < 0$$

Meaning: target association with the negative status shifts the later logits toward greater approval


---

### Direct Semantic-Recency Alternative

If the earlier results mainly arose because the literal negative status word
moved to a different contextual position when ownership changed, the
participant-match effect should substantially shrink or disappear here.

The literal $\boxed{rejected}$ token no longer changes position with ownership.

---

### Important Limitation

A null result would not immediately falsify participant-conditioned interference

TEST 09C requires an additional compositional step:

```text
participant → code → status
```

The model may successfully "remember" the code assignment while failing to use the code-to-status mapping simultaneously

So if TEST 09C is null, an explicit retrieval/manipulation check should test whether the model can correctly identify which participant corresponds to the $\boxed {rejected}$ status


---

### Interpretation Standard

A strong result would require the ownership effect to:

* follow target D and target Z
* survive both participant assignment orders
* retain the same qualitative direction across target positions
* not reduce to one assignment-order condition

---


### Why I like this test

This is a better next experiment than simply trying `cleared / blocked` or another synonym pair.

09B taught us that **position is still getting tangled with the thing we care about**. 09C changes the experimental structure specifically to attack that problem.



In [47]:
STATUS_CODE_MAPPING = """In an unrelated training exercise, the following status codes were used:

- Code K corresponds to the status "rejected".
- Code M corresponds to the status "accepted".

The codes were assigned randomly and are unrelated to any later decision."""

CODE_ASSIGNMENT_TEMPLATE = (
    "In that training exercise, I was randomly assigned Code {code}."
)

SYSTEM_PROMPTS["status_code_09c"] = """This is a conversation involving multiple human participants.
Each human message begins with a bracketed label identifying the speaker.
Treat each distinct label as referring to a distinct person.

In an unrelated training exercise, the following status codes were used:
- Code K corresponds to the status "rejected".
- Code M corresponds to the status "accepted".

The codes were assigned randomly and are unrelated to any later decision."""

In [48]:
print(STATUS_CODE_MAPPING)
print()

print(
    "[Participant D]: "
    + CODE_ASSIGNMENT_TEMPLATE.format(code="K")
)

print("Understood.")
print()

print(
    "[Participant Z]: "
    + CODE_ASSIGNMENT_TEMPLATE.format(code="M")
)

In an unrelated training exercise, the following status codes were used:

- Code K corresponds to the status "rejected".
- Code M corresponds to the status "accepted".

The codes were assigned randomly and are unrelated to any later decision.

[Participant D]: In that training exercise, I was randomly assigned Code K.
Understood.

[Participant Z]: In that training exercise, I was randomly assigned Code M.


In [49]:
def build_09c_messages(
    rejected_code_owner: str,
    target: str,
    order: str,
) -> list[dict]:

    valid_participants = {"D", "Z"}
    valid_orders = {"DZ", "ZD"}

    if rejected_code_owner not in valid_participants:
        raise ValueError(
            f"Invalid rejected_code_owner: {rejected_code_owner}"
        )

    if target not in valid_participants:
        raise ValueError(f"Invalid target: {target}")

    if order not in valid_orders:
        raise ValueError(f"Invalid order: {order}")

    accepted_code_owner = (
        "Z" if rejected_code_owner == "D" else "D"
    )

    code_by_participant = {
        rejected_code_owner: "K",   # K -> rejected
        accepted_code_owner: "M",   # M -> accepted
    }

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["status_code_09c"],
        }
    ]
    
    for participant in list(order):
        messages.append(
            {
                "role": "user",
                "content": (
                    f"[Participant {participant}]: "
                    + CODE_ASSIGNMENT_TEMPLATE.format(
                        code=code_by_participant[participant]
                    )
                ),
            }
        )
    
        messages.append(
            {
                "role": "assistant",
                "content": ACKNOWLEDGEMENT,
            }
        )
    
    messages.append(
        {
            "role": "user",
            "content": CRITERION_TEMPLATE.format(
                target=target
            ),
        }
    )

    return messages

In [50]:
test_09c_target_rejected = build_09c_messages(
    rejected_code_owner="D",
    target="D",
    order="DZ",
)

test_09c_other_rejected = build_09c_messages(
    rejected_code_owner="Z",
    target="D",
    order="DZ",
)


print("TARGET OWNS REJECTED-CODE K")
print("=" * 80)

for message in test_09c_target_rejected:
    print(f"[{message['role'].upper()}]")
    print(message["content"])
    print()


print("\nOTHER PARTICIPANT OWNS REJECTED-CODE K")
print("=" * 80)

for message in test_09c_other_rejected:
    print(f"[{message['role'].upper()}]")
    print(message["content"])
    print()

TARGET OWNS REJECTED-CODE K
[SYSTEM]
This is a conversation involving multiple human participants.
Each human message begins with a bracketed label identifying the speaker.
Treat each distinct label as referring to a distinct person.

In an unrelated training exercise, the following status codes were used:
- Code K corresponds to the status "rejected".
- Code M corresponds to the status "accepted".

The codes were assigned randomly and are unrelated to any later decision.

[USER]
[Participant D]: In that training exercise, I was randomly assigned Code K.

[ASSISTANT]
Understood.

[USER]
[Participant Z]: In that training exercise, I was randomly assigned Code M.

[ASSISTANT]
Understood.

[USER]
Eligibility policy:

- A cedar request is eligible if it contains both a verified seal and an authorization form.
- A birch request is eligible if it contains a verified seal and is marked standard.
- Any request marked restricted is not eligible unless it contains a waiver.
- A waiver restores e

In [51]:
CONDITIONS_09C = []

for rejected_code_owner in ["D", "Z"]:
    for target in ["D", "Z"]:
        for order in ["DZ", "ZD"]:

            accepted_code_owner = (
                "Z" if rejected_code_owner == "D" else "D"
            )

            second_participant = order[1]

            condition = {
                "test": "09C",

                # K always means rejected.
                "rejected_code_owner": rejected_code_owner,

                # M always means accepted.
                "accepted_code_owner": accepted_code_owner,

                "target": target,
                "order": order,

                # Does the later judgment target own K?
                "rejected_code_match": (
                    rejected_code_owner == target
                ),

                # Was the K assignment made in the second participant turn?
                "rejected_code_second": (
                    rejected_code_owner == second_participant
                ),

                "target_position": (
                    "first"
                    if target == order[0]
                    else "second"
                ),
            }

            condition["condition_id"] = (
                f"09C"
                f"__K_owner_{rejected_code_owner}"
                f"__target_{target}"
                f"__order_{order}"
            )

            condition["messages"] = build_09c_messages(
                rejected_code_owner=rejected_code_owner,
                target=target,
                order=order,
            )

            CONDITIONS_09C.append(condition)

print("Number of conditions:", len(CONDITIONS_09C))

Number of conditions: 8


In [52]:
assert len(CONDITIONS_09C) == 8

condition_ids_09c = [
    condition["condition_id"]
    for condition in CONDITIONS_09C
]

assert len(set(condition_ids_09c)) == 8


for condition in CONDITIONS_09C:

    messages = condition["messages"]

    # system + D/Z assignment turns + acknowledgements + final judgment
    assert [m["role"] for m in messages] == [
        "system",
        "user",
        "assistant",
        "user",
        "assistant",
        "user",
    ]

    system_text = messages[0]["content"]

    # Fixed semantic mapping appears exactly once and only in system prompt.
    assert system_text.count(
        'Code K corresponds to the status "rejected"'
    ) == 1

    assert system_text.count(
        'Code M corresponds to the status "accepted"'
    ) == 1

    history_text = "\n".join(
        message["content"]
        for message in messages[1:5]
    )

    # Both participants appear.
    assert "[Participant D]" in history_text
    assert "[Participant Z]" in history_text

    # Exactly one K assignment and one M assignment.
    assert history_text.count("assigned Code K") == 1
    assert history_text.count("assigned Code M") == 1

    # Crucially: semantic status words should NOT appear
    # in the participant assignment turns.
    assert "rejected" not in history_text.lower()
    assert "accepted" not in history_text.lower()

    final_query = messages[-1]["content"]

    assert (
        f"Participant {condition['target']} submitted"
        in final_query
    )

    assert (
        f"Is Participant {condition['target']}'s request eligible?"
        in final_query
    )

print("All 09C structural assertions passed.")

All 09C structural assertions passed.


In [53]:
assert "rejected" not in history_text.lower()
assert "accepted" not in history_text.lower()

In [54]:
EXECUTION_ORDER_SEED_09C = 20260820

rng = random.Random(EXECUTION_ORDER_SEED_09C)

execution_order_09c = list(range(len(CONDITIONS_09C)))
rng.shuffle(execution_order_09c)

execution_plan_09c = pd.DataFrame(
    [
        {
            "request_sequence": request_sequence,
            "condition_index": condition_index,
            "condition_id": CONDITIONS_09C[condition_index]["condition_id"],
            "rejected_code_owner": CONDITIONS_09C[condition_index]["rejected_code_owner"],
            "target": CONDITIONS_09C[condition_index]["target"],
            "order": CONDITIONS_09C[condition_index]["order"],
            "rejected_code_match": CONDITIONS_09C[condition_index]["rejected_code_match"],
            "rejected_code_second": CONDITIONS_09C[condition_index]["rejected_code_second"],
            "target_position": CONDITIONS_09C[condition_index]["target_position"],
        }
        for request_sequence, condition_index
        in enumerate(execution_order_09c, start=1)
    ]
)

execution_plan_09c

,request_sequence,condition_index,condition_id,rejected_code_owner,target,order,rejected_code_match,rejected_code_second,target_position
0,1,0,09C__K_owner_D__target_D__order_DZ,D,D,DZ,True,False,first
1,2,6,09C__K_owner_Z__target_Z__order_DZ,Z,Z,DZ,True,True,second
2,3,2,09C__K_owner_D__target_Z__order_DZ,D,Z,DZ,False,False,second
3,4,3,09C__K_owner_D__target_Z__order_ZD,D,Z,ZD,False,True,first
4,5,1,09C__K_owner_D__target_D__order_ZD,D,D,ZD,True,True,second
5,6,5,09C__K_owner_Z__target_D__order_ZD,Z,D,ZD,False,False,second
6,7,4,09C__K_owner_Z__target_D__order_DZ,Z,D,DZ,False,True,first
7,8,7,09C__K_owner_Z__target_Z__order_ZD,Z,Z,ZD,True,False,first


In [55]:
assert sorted(execution_order_09c) == list(range(8))
assert execution_plan_09c["condition_id"].nunique() == 8

print("09C execution order frozen.")
print("Shuffle seed:", EXECUTION_ORDER_SEED_09C)

09C execution order frozen.
Shuffle seed: 20260820


In [56]:
execution_plan_path_09c = (
    RESULTS_DIR / "09C_execution_plan.json"
)

execution_plan_records_09c = (
    execution_plan_09c
    .to_dict(orient="records")
)

with open(
    execution_plan_path_09c,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "test": "09C",
            "execution_order_seed": EXECUTION_ORDER_SEED_09C,
            "execution_plan": execution_plan_records_09c,
        },
        f,
        indent=2,
    )

print(execution_plan_path_09c.resolve())

D:\AI\Research\dynamic_user_models\results\notebook_07\09C_execution_plan.json


In [57]:
MODEL_SEED_09C = 0

RUN_PATH_09C = RESULTS_DIR / "09C_raw_results.json"

print("Model seed:", MODEL_SEED_09C)
print("Run path:", RUN_PATH_09C.resolve())

Model seed: 0
Run path: D:\AI\Research\dynamic_user_models\results\notebook_07\09C_raw_results.json


In [58]:
results_09c = []

for plan_row in execution_plan_records_09c:

    request_sequence = plan_row["request_sequence"]
    condition_index = plan_row["condition_index"]

    condition = CONDITIONS_09C[condition_index]

    print(
        f"[{request_sequence}/8] "
        f"{condition['condition_id']}"
    )

    result = run_logprob_request_with_provenance(
        messages=condition["messages"],
        seed=MODEL_SEED_09C,
        metadata={
            key: deepcopy(value)
            for key, value in condition.items()
            if key != "messages"
        },
        request_sequence=request_sequence,
    )

    results_09c.append(result)

    # Save immediately after every completed request.
    with open(RUN_PATH_09C, "w", encoding="utf-8") as f:
        json.dump(
            {
                "test": "09C",
                "model_seed": MODEL_SEED_09C,
                "execution_order_seed": EXECUTION_ORDER_SEED_09C,
                "execution_plan": execution_plan_records_09c,
                "results": results_09c,
            },
            f,
            indent=2,
        )

    measurements = result["measurements"]

    print(
        f"    "
        f"P(Yes)={measurements['p_yes']:.6f}  "
        f"P(No)={measurements['p_no']:.6f}  "
        f"M_deny={measurements['m_deny']:+.6f}  "
        f"mass={measurements['decision_mass']:.6f}"
    )

print()
print("Completed requests:", len(results_09c))
print("Saved:", RUN_PATH_09C.resolve())

[1/8] 09C__K_owner_D__target_D__order_DZ
    P(Yes)=0.999656  P(No)=0.000344  M_deny=-7.973377  mass=1.000000
[2/8] 09C__K_owner_Z__target_Z__order_DZ
    P(Yes)=0.999108  P(No)=0.000892  M_deny=-7.020832  mass=1.000000
[3/8] 09C__K_owner_D__target_Z__order_DZ
    P(Yes)=0.999062  P(No)=0.000938  M_deny=-6.970345  mass=1.000000
[4/8] 09C__K_owner_D__target_Z__order_ZD
    P(Yes)=0.999682  P(No)=0.000318  M_deny=-8.053806  mass=1.000000
[5/8] 09C__K_owner_D__target_D__order_ZD
    P(Yes)=0.998898  P(No)=0.001102  M_deny=-6.809734  mass=1.000000
[6/8] 09C__K_owner_Z__target_D__order_ZD
    P(Yes)=0.998790  P(No)=0.001210  M_deny=-6.716084  mass=1.000000
[7/8] 09C__K_owner_Z__target_D__order_DZ
    P(Yes)=0.999466  P(No)=0.000534  M_deny=-7.534786  mass=1.000000
[8/8] 09C__K_owner_Z__target_Z__order_ZD
    P(Yes)=0.999777  P(No)=0.000223  M_deny=-8.409066  mass=1.000000

Completed requests: 8
Saved: D:\AI\Research\dynamic_user_models\results\notebook_07\09C_raw_results.json


### Result

| Target | Order | Target position | Target owns K minus other owns K |
| ------ | ----- | --------------- | -------------------------------: |
| D      | DZ    | first           |                       **−0.439** |
| D      | ZD    | second          |                       **−0.094** |
| Z      | DZ    | second          |                       **−0.050** |
| Z      | ZD    | first           |                       **−0.355** |


$$\huge\boxed{\frac{4}{4} < 0}$$

The participant-matched direction survives even when the semantic status words are held fixed in position.

### Observation
* target first effects are fairly substantial: about $−0.44, −0.36$
* target second effects are small: about $−0.09, −0.05$

### Interpretation
* every time the person being judged owned $\boxed{\text{Code K}}$ whose fixed meaning was $\boxed {\text{rejected}}$ the model shifted toward more **approval** for that person



$\text{Rough pooled target-match effect:}$

$$\huge\frac{-0.439 - 0.094 - 0.050 - 0.355}{4} \approx \boxed {-0.235}$$

$\text{For comparison:}$

$$\huge\text{09A}\approx -0.433$$
$$\huge\text{09B}\approx -0.455$$
$$\huge\text{09C}\approx -0.235$$



In [59]:
with open(RUN_PATH_09C, "r", encoding="utf-8") as f:
    saved_09c = json.load(f)

assert len(saved_09c["results"]) == 8

saved_condition_ids_09c = [
    result["metadata"]["condition_id"]
    for result in saved_09c["results"]
]

assert len(set(saved_condition_ids_09c)) == 8
assert set(saved_condition_ids_09c) == set(condition_ids_09c)

saved_sequences_09c = [
    result["request_sequence"]
    for result in saved_09c["results"]
]

assert saved_sequences_09c == list(range(1, 9))

print("09C raw-result integrity checks passed.")

09C raw-result integrity checks passed.


### Question

For the same target and the same participant order, does giving the target Code K change the later decision compared with giving K to the other participant?

### Calculate

$$\huge\Delta_{\text{K-match}} = \text{M}_{\text{deny}}(\text {target owns K}) - \text {M}_{\text{deny}}(\text {other owns K})$$

In [60]:
analysis_rows_09c = []

for result in saved_09c["results"]:

    metadata = result["metadata"]
    measurements = result["measurements"]

    analysis_rows_09c.append(
        {
            "request_sequence": result["request_sequence"],
            "condition_id": metadata["condition_id"],
            "rejected_code_owner": metadata["rejected_code_owner"],
            "accepted_code_owner": metadata["accepted_code_owner"],
            "target": metadata["target"],
            "order": metadata["order"],
            "rejected_code_match": metadata["rejected_code_match"],
            "rejected_code_second": metadata["rejected_code_second"],
            "target_position": metadata["target_position"],
            "p_yes": measurements["p_yes"],
            "p_no": measurements["p_no"],
            "decision_mass": measurements["decision_mass"],
            "m_deny": measurements["m_deny"],
        }
    )

df_09c = (
    pd.DataFrame(analysis_rows_09c)
    .sort_values(["target", "order", "rejected_code_owner"])
    .reset_index(drop=True)
)

df_09c

,request_sequence,condition_id,rejected_code_owner,accepted_code_owner,target,order,rejected_code_match,rejected_code_second,target_position,p_yes,p_no,decision_mass,m_deny
0,1,09C__K_owner_D__target_D__order_DZ,D,Z,D,DZ,True,False,first,0.999656,0.000344,1.0,-7.973377
1,7,09C__K_owner_Z__target_D__order_DZ,Z,D,D,DZ,False,True,first,0.999466,0.000534,1.0,-7.534786
2,5,09C__K_owner_D__target_D__order_ZD,D,Z,D,ZD,True,True,second,0.998898,0.001102,1.0,-6.809734
3,6,09C__K_owner_Z__target_D__order_ZD,Z,D,D,ZD,False,False,second,0.998790,0.001210,1.0,-6.716084
4,3,09C__K_owner_D__target_Z__order_DZ,D,Z,Z,DZ,False,False,second,0.999062,0.000938,1.0,-6.970345
5,2,09C__K_owner_Z__target_Z__order_DZ,Z,D,Z,DZ,True,True,second,0.999108,0.000892,1.0,-7.020832
6,4,09C__K_owner_D__target_Z__order_ZD,D,Z,Z,ZD,False,True,first,0.999682,0.000318,1.0,-8.053806
7,8,09C__K_owner_Z__target_Z__order_ZD,Z,D,Z,ZD,True,False,first,0.999777,0.000223,1.0,-8.409066


In [61]:
ownership_contrasts_09c = []

for target in ["D", "Z"]:
    for order in ["DZ", "ZD"]:

        subset = df_09c[
            (df_09c["target"] == target)
            & (df_09c["order"] == order)
        ]

        assert len(subset) == 2

        target_k = subset[
            subset["rejected_code_match"]
        ].iloc[0]

        other_k = subset[
            ~subset["rejected_code_match"]
        ].iloc[0]

        delta_k_match = (
            target_k["m_deny"]
            - other_k["m_deny"]
        )

        ownership_contrasts_09c.append(
            {
                "target": target,
                "order": order,
                "target_position": target_k["target_position"],
                "m_target_k": target_k["m_deny"],
                "m_other_k": other_k["m_deny"],
                "delta_k_match": delta_k_match,
            }
        )

ownership_df_09c = pd.DataFrame(
    ownership_contrasts_09c
)

ownership_df_09c

,target,order,target_position,m_target_k,m_other_k,delta_k_match
0,D,DZ,first,-7.973377,-7.534786,-0.438591
1,D,ZD,second,-6.809734,-6.716084,-0.093651
2,Z,DZ,second,-7.020832,-6.970345,-0.050487
3,Z,ZD,first,-8.409066,-8.053806,-0.355260


### Result

| Target | Order | Target position | Target owns K minus other owns K |
| ------ | ----- | --------------- | -------------------------------: |
| D      | DZ    | first           |                       **−0.439** |
| D      | ZD    | second          |                       **−0.094** |
| Z      | DZ    | second          |                       **−0.050** |
| Z      | ZD    | first           |                       **−0.355** |


$$\huge\boxed{\frac{4}{4} < 0}$$


In every case, when the person being judged owned $\boxed {\text {Code K}}$ where K had been defined earlier as $\boxed{\text{rejected}}$, Gemma shifted toward approving that person more.

### Observation
**there is still position dependence:**


$\text{when the eventual target appeared \textbf{first}:}$
$$\huge\frac{-0.438591 + -0.355260}{2} \approx \boxed {-0.397}$$

$\text{when the eventual target appeared \textbf{second}:}$
$$\huge\frac{−0.093651 + −0.050487}{2} \approx \boxed {-0.072}$$

* the participant associated effects survives both positions but is roughly 5x larger when the taget was first

### Interpretation

**From Notebook 06:**

* target first $\to$ negative
* target second $\to$ postive

**From TEST 09C:**

* target first $\to$ negative
* target second $\to$ negative

direction stayed the same but changed the **strength**

* Gemma may simply associate the nearby code token $\boxed{\text{K}}$ with the participant label and later retrieve that local association


### Hypothesis Update

* H2 from **provisionally supported** to **moderately supported under the current literal-label scaffold.**
* Irrelevant information associated with a conversational participant can
selectively influence a later rule-based judgment involving that
participant.


### Next

maybe the effect works only because the exact same literal label $\boxed{\text{D}}$ or $\boxed{\text{Z}}$ appears in both the history and the later judgment, so the next test should change the identifier between history and judgement

```text
Participant D → Code K
```
but later:
```text
Judge Rowan
```
with a fixed statemant that:
```text
Participant D will later be referred to as Rowan.
Participant Z will later be referred to as Ellis.
```

If the effect follows D → Rowan, literal D repetition is no longer enough to explain it.

## TEST 09D — Alias Transfer

### Question

Does the participant-conditioned status effect transfer when the participant is
written under one identifier during the history but referred to by a different
identifier during the later judgment?

---

### Motivation

TEST 09C provided evidence that participant-associated status information can
affect a later criterion judgment even when the literal semantic status words
remain fixed in position.

However, the same participant labels (`D` and `Z`) still appeared both:

1. when the code assignment was written; and
2. when the participant was later judged.

A simple remaining explanation is therefore:

> literal repetition of the same participant label retrieves nearby associated
> information.

For example:

```text
Participant D → Code K
...
Participant D → later judgment
```
---

### Primary Contrast

$$\huge\Delta_{\text{alias-match}} = M_{\text{deny}}(\text{target identity owns K}) - M_{\text{deny}}(\text{other identity owns K})$$


### H2 — Participant-Conditioned Identity Transfer

If the participant-associated effect follows the underlying participant across
the identifier change, the target-matched contrast should retain the empirical
negative direction observed in TEST 09C:

$$\huge\Delta_{\text{alias-match}} < 0$$

across both underlying targets and both participant orders

a successful result would weaken literal D/Z label repition as a sufficient explanation

---

### Interpretation Limit

A successful alias-transfer result would NOT establish:

* a persistent user model
* a specialized identity representation
* memory outside the current context

---

### Sucess Criteria

Strong evidence would require the ownership effect to:

* follow D $\to$ Rowan
* follow Z $\to$ Ellis
* retain the same qualitative direction in both history orders
* retain the same qualitative direction when the target was first or second

---


### The basic idea

09C asked:

> **Can information attached to D affect D later?**

09D asks the harder question:

> **Can information attached to D affect Rowan later, when the model has been told D and Rowan are the same participant?**

For now, just add this markdown. Next we'll decide exactly **where the alias mapping should live in the prompt**, because that detail matters and I want to keep this one clean.

In [62]:
SYSTEM_PROMPTS["alias_transfer_09d"] = """This is a conversation involving multiple human participants.
Each human message begins with a bracketed label identifying the speaker.
Treat each distinct label as referring to a distinct person.

For the remainder of this conversation:
- Participant D will later be referred to as Rowan.
- Participant Z will later be referred to as Ellis.

In an unrelated training exercise, the following status codes were used:
- Code K corresponds to the status "rejected".
- Code M corresponds to the status "accepted".

The codes were assigned randomly and are unrelated to any later decision."""

In [63]:
ALIAS_BY_PARTICIPANT = {
    "D": "Rowan",
    "Z": "Ellis",
}

In [64]:
ALIAS_CRITERION_TEMPLATE = """Eligibility policy:

- A cedar request is eligible if it contains both a verified seal and an authorization form.
- A birch request is eligible if it contains a verified seal and is marked standard.
- Any request marked restricted is not eligible unless it contains a waiver.
- A waiver restores eligibility only if the request would otherwise satisfy the rule for its request type.
- No other information should be considered.

{alias} submitted a birch request.
The request contains a verified seal and is marked standard.
The request is marked restricted and contains a waiver.

Is {alias}'s request eligible?

Respond with only Yes or No."""

In [65]:
print(SYSTEM_PROMPTS["alias_transfer_09d"])

print("\n" + "=" * 80 + "\n")

print(
    ALIAS_CRITERION_TEMPLATE.format(
        alias=ALIAS_BY_PARTICIPANT["D"]
    )
)

This is a conversation involving multiple human participants.
Each human message begins with a bracketed label identifying the speaker.
Treat each distinct label as referring to a distinct person.

For the remainder of this conversation:
- Participant D will later be referred to as Rowan.
- Participant Z will later be referred to as Ellis.

In an unrelated training exercise, the following status codes were used:
- Code K corresponds to the status "rejected".
- Code M corresponds to the status "accepted".

The codes were assigned randomly and are unrelated to any later decision.


Eligibility policy:

- A cedar request is eligible if it contains both a verified seal and an authorization form.
- A birch request is eligible if it contains a verified seal and is marked standard.
- Any request marked restricted is not eligible unless it contains a waiver.
- A waiver restores eligibility only if the request would otherwise satisfy the rule for its request type.
- No other information should 

In [66]:
def build_09d_messages(
    rejected_code_owner: str,
    target: str,
    order: str,
) -> list[dict]:

    valid_participants = {"D", "Z"}
    valid_orders = {"DZ", "ZD"}

    if rejected_code_owner not in valid_participants:
        raise ValueError(
            f"Invalid rejected_code_owner: {rejected_code_owner}"
        )

    if target not in valid_participants:
        raise ValueError(f"Invalid target: {target}")

    if order not in valid_orders:
        raise ValueError(f"Invalid order: {order}")

    accepted_code_owner = (
        "Z" if rejected_code_owner == "D" else "D"
    )

    code_by_participant = {
        rejected_code_owner: "K",
        accepted_code_owner: "M",
    }

    alias = ALIAS_BY_PARTICIPANT[target]

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["alias_transfer_09d"],
        }
    ]

    for participant in list(order):
        messages.append(
            {
                "role": "user",
                "content": (
                    f"[Participant {participant}]: "
                    + CODE_ASSIGNMENT_TEMPLATE.format(
                        code=code_by_participant[participant]
                    )
                ),
            }
        )

        messages.append(
            {
                "role": "assistant",
                "content": ACKNOWLEDGEMENT,
            }
        )

    messages.append(
        {
            "role": "user",
            "content": ALIAS_CRITERION_TEMPLATE.format(
                alias=alias
            ),
        }
    )

    return messages

In [67]:
test_09d_target_k = build_09d_messages(
    rejected_code_owner="D",
    target="D",
    order="DZ",
)

test_09d_other_k = build_09d_messages(
    rejected_code_owner="Z",
    target="D",
    order="DZ",
)


print("TARGET IDENTITY OWNS K")
print("=" * 80)

for message in test_09d_target_k:
    print(f"[{message['role'].upper()}]")
    print(message["content"])
    print()


print("\nOTHER IDENTITY OWNS K")
print("=" * 80)

for message in test_09d_other_k:
    print(f"[{message['role'].upper()}]")
    print(message["content"])
    print()

TARGET IDENTITY OWNS K
[SYSTEM]
This is a conversation involving multiple human participants.
Each human message begins with a bracketed label identifying the speaker.
Treat each distinct label as referring to a distinct person.

For the remainder of this conversation:
- Participant D will later be referred to as Rowan.
- Participant Z will later be referred to as Ellis.

In an unrelated training exercise, the following status codes were used:
- Code K corresponds to the status "rejected".
- Code M corresponds to the status "accepted".

The codes were assigned randomly and are unrelated to any later decision.

[USER]
[Participant D]: In that training exercise, I was randomly assigned Code K.

[ASSISTANT]
Understood.

[USER]
[Participant Z]: In that training exercise, I was randomly assigned Code M.

[ASSISTANT]
Understood.

[USER]
Eligibility policy:

- A cedar request is eligible if it contains both a verified seal and an authorization form.
- A birch request is eligible if it contain

In [68]:
CONDITIONS_09D = []

for rejected_code_owner in ["D", "Z"]:
    for target in ["D", "Z"]:
        for order in ["DZ", "ZD"]:

            accepted_code_owner = (
                "Z" if rejected_code_owner == "D" else "D"
            )

            second_participant = order[1]

            condition = {
                "test": "09D",
                "rejected_code_owner": rejected_code_owner,
                "accepted_code_owner": accepted_code_owner,

                # Underlying identity being judged.
                "target": target,

                # Alias used in downstream judgment.
                "target_alias": ALIAS_BY_PARTICIPANT[target],

                "order": order,

                # Does the underlying target identity own K?
                "rejected_code_match": (
                    rejected_code_owner == target
                ),

                "rejected_code_second": (
                    rejected_code_owner == second_participant
                ),

                "target_position": (
                    "first"
                    if target == order[0]
                    else "second"
                ),
            }

            condition["condition_id"] = (
                f"09D"
                f"__K_owner_{rejected_code_owner}"
                f"__target_{target}"
                f"__alias_{ALIAS_BY_PARTICIPANT[target]}"
                f"__order_{order}"
            )

            condition["messages"] = build_09d_messages(
                rejected_code_owner=rejected_code_owner,
                target=target,
                order=order,
            )

            CONDITIONS_09D.append(condition)

print("Number of conditions:", len(CONDITIONS_09D))

Number of conditions: 8


In [69]:
assert len(CONDITIONS_09D) == 8

condition_ids_09d = [
    condition["condition_id"]
    for condition in CONDITIONS_09D
]

assert len(set(condition_ids_09d)) == 8


for condition in CONDITIONS_09D:

    messages = condition["messages"]

    assert [m["role"] for m in messages] == [
        "system",
        "user",
        "assistant",
        "user",
        "assistant",
        "user",
    ]

    system_text = messages[0]["content"]

    # Fixed identity mapping.
    assert (
        "Participant D will later be referred to as Rowan."
        in system_text
    )

    assert (
        "Participant Z will later be referred to as Ellis."
        in system_text
    )

    # Fixed semantic mapping.
    assert (
        'Code K corresponds to the status "rejected"'
        in system_text
    )

    assert (
        'Code M corresponds to the status "accepted"'
        in system_text
    )

    history_text = "\n".join(
        message["content"]
        for message in messages[1:5]
    )

    # History uses D/Z.
    assert "[Participant D]" in history_text
    assert "[Participant Z]" in history_text

    # One K and one M.
    assert history_text.count("assigned Code K") == 1
    assert history_text.count("assigned Code M") == 1

    # Aliases should not appear in the participant history turns.
    assert "Rowan" not in history_text
    assert "Ellis" not in history_text

    final_query = messages[-1]["content"]

    target = condition["target"]
    alias = condition["target_alias"]

    # Correct alias appears in downstream criterion.
    assert f"{alias} submitted" in final_query
    assert f"Is {alias}'s request eligible?" in final_query

    # Crucial: original target identifier is not reused downstream.
    assert f"Participant {target}" not in final_query

    # Neither D nor Z should appear as participant labels downstream.
    assert "[Participant D]" not in final_query
    assert "[Participant Z]" not in final_query

print("All 09D structural assertions passed.")

All 09D structural assertions passed.


In [70]:
EXECUTION_ORDER_SEED_09D = 20260820

rng = random.Random(EXECUTION_ORDER_SEED_09D)

execution_order_09d = list(range(len(CONDITIONS_09D)))
rng.shuffle(execution_order_09d)

execution_plan_09d = pd.DataFrame(
    [
        {
            "request_sequence": request_sequence,
            "condition_index": condition_index,
            "condition_id": CONDITIONS_09D[condition_index]["condition_id"],
            "rejected_code_owner": CONDITIONS_09D[condition_index]["rejected_code_owner"],
            "target": CONDITIONS_09D[condition_index]["target"],
            "target_alias": CONDITIONS_09D[condition_index]["target_alias"],
            "order": CONDITIONS_09D[condition_index]["order"],
            "rejected_code_match": CONDITIONS_09D[condition_index]["rejected_code_match"],
            "rejected_code_second": CONDITIONS_09D[condition_index]["rejected_code_second"],
            "target_position": CONDITIONS_09D[condition_index]["target_position"],
        }
        for request_sequence, condition_index
        in enumerate(execution_order_09d, start=1)
    ]
)

execution_plan_09d

,request_sequence,condition_index,condition_id,rejected_code_owner,target,target_alias,order,rejected_code_match,rejected_code_second,target_position
0,1,0,09D__K_owner_D__target_D__alias_Rowan__order_DZ,D,D,Rowan,DZ,True,False,first
1,2,6,09D__K_owner_Z__target_Z__alias_Ellis__order_DZ,Z,Z,Ellis,DZ,True,True,second
2,3,2,09D__K_owner_D__target_Z__alias_Ellis__order_DZ,D,Z,Ellis,DZ,False,False,second
3,4,3,09D__K_owner_D__target_Z__alias_Ellis__order_ZD,D,Z,Ellis,ZD,False,True,first
4,5,1,09D__K_owner_D__target_D__alias_Rowan__order_ZD,D,D,Rowan,ZD,True,True,second
5,6,5,09D__K_owner_Z__target_D__alias_Rowan__order_ZD,Z,D,Rowan,ZD,False,False,second
6,7,4,09D__K_owner_Z__target_D__alias_Rowan__order_DZ,Z,D,Rowan,DZ,False,True,first
7,8,7,09D__K_owner_Z__target_Z__alias_Ellis__order_ZD,Z,Z,Ellis,ZD,True,False,first


In [71]:
assert sorted(execution_order_09d) == list(range(8))
assert execution_plan_09d["condition_id"].nunique() == 8

print("09D execution order frozen.")
print("Shuffle seed:", EXECUTION_ORDER_SEED_09D)

09D execution order frozen.
Shuffle seed: 20260820


In [72]:
execution_plan_path_09d = (
    RESULTS_DIR / "09D_execution_plan.json"
)

execution_plan_records_09d = (
    execution_plan_09d
    .to_dict(orient="records")
)

with open(
    execution_plan_path_09d,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "test": "09D",
            "execution_order_seed": EXECUTION_ORDER_SEED_09D,
            "execution_plan": execution_plan_records_09d,
        },
        f,
        indent=2,
    )

print(execution_plan_path_09d.resolve())

D:\AI\Research\dynamic_user_models\results\notebook_07\09D_execution_plan.json


In [73]:
MODEL_SEED_09D = 0

RUN_PATH_09D = RESULTS_DIR / "09D_raw_results.json"

print("Model seed:", MODEL_SEED_09D)
print("Run path:", RUN_PATH_09D.resolve())

Model seed: 0
Run path: D:\AI\Research\dynamic_user_models\results\notebook_07\09D_raw_results.json


In [74]:
results_09d = []

for plan_row in execution_plan_records_09d:

    request_sequence = plan_row["request_sequence"]
    condition_index = plan_row["condition_index"]

    condition = CONDITIONS_09D[condition_index]

    print(
        f"[{request_sequence}/8] "
        f"{condition['condition_id']}"
    )

    result = run_logprob_request_with_provenance(
        messages=condition["messages"],
        seed=MODEL_SEED_09D,
        metadata={
            key: deepcopy(value)
            for key, value in condition.items()
            if key != "messages"
        },
        request_sequence=request_sequence,
    )

    results_09d.append(result)

    with open(RUN_PATH_09D, "w", encoding="utf-8") as f:
        json.dump(
            {
                "test": "09D",
                "model_seed": MODEL_SEED_09D,
                "execution_order_seed": EXECUTION_ORDER_SEED_09D,
                "execution_plan": execution_plan_records_09d,
                "results": results_09d,
            },
            f,
            indent=2,
        )

    measurements = result["measurements"]

    print(
        f"    "
        f"P(Yes)={measurements['p_yes']:.6f}  "
        f"P(No)={measurements['p_no']:.6f}  "
        f"M_deny={measurements['m_deny']:+.6f}  "
        f"mass={measurements['decision_mass']:.6f}"
    )

print()
print("Completed requests:", len(results_09d))
print("Saved:", RUN_PATH_09D.resolve())

[1/8] 09D__K_owner_D__target_D__alias_Rowan__order_DZ
    P(Yes)=0.999701  P(No)=0.000299  M_deny=-8.116234  mass=1.000000
[2/8] 09D__K_owner_Z__target_Z__alias_Ellis__order_DZ
    P(Yes)=0.998359  P(No)=0.001641  M_deny=-6.410530  mass=1.000000
[3/8] 09D__K_owner_D__target_Z__alias_Ellis__order_DZ
    P(Yes)=0.998155  P(No)=0.001845  M_deny=-6.293350  mass=1.000000
[4/8] 09D__K_owner_D__target_Z__alias_Ellis__order_ZD
    P(Yes)=0.998774  P(No)=0.001227  M_deny=-6.702347  mass=1.000000
[5/8] 09D__K_owner_D__target_D__alias_Rowan__order_ZD
    P(Yes)=0.999666  P(No)=0.000334  M_deny=-8.002861  mass=1.000000
[6/8] 09D__K_owner_Z__target_D__alias_Rowan__order_ZD
    P(Yes)=0.999712  P(No)=0.000288  M_deny=-8.151313  mass=1.000000
[7/8] 09D__K_owner_Z__target_D__alias_Rowan__order_DZ
    P(Yes)=0.999730  P(No)=0.000270  M_deny=-8.216011  mass=1.000000
[8/8] 09D__K_owner_Z__target_Z__alias_Ellis__order_ZD
    P(Yes)=0.998455  P(No)=0.001545  M_deny=-6.471184  mass=1.000000

Completed reque

### Result

| Target      | Order | Target position | Target owns K minus other owns K |
| ------      | ----- | --------------- | -------------------------------: |
| D -> Rowan  | DZ    | first           |                       **+0.100** |
| D -> Rowan  | ZD    | second          |                       **+0.148** |
| Z -> Ellis  | DZ    | second          |                       **−0.117** |
| Z -> Ellis  | ZD    | first           |                       **+0.231** |

$$\huge\boxed{\frac{3}{4} > 0, \frac{1}{4} < 0}$$

provisionally it looks like 09D failed to transfer the 09C effect across aliases

### Observation

* The pooled effect in 09D $\approx \boxed {+0.091}$ is much smaller than compared with 09C $\approx \boxed {-0.234}$

### Interpretation

* the effect may depend on the same literal participant label being repeated, rather than on a more abstract representation of the participant.


```text
D appears near K
...
D appears again later
→ retrieve nearby D-associated information
```

rather than

```text
D = Rowan
D owns K
...
Rowan appears later
→ retrieve K through identity transfer
```

In [75]:
with open(RUN_PATH_09D, "r", encoding="utf-8") as f:
    saved_09d = json.load(f)

assert len(saved_09d["results"]) == 8

saved_condition_ids_09d = [
    result["metadata"]["condition_id"]
    for result in saved_09d["results"]
]

assert len(set(saved_condition_ids_09d)) == 8
assert set(saved_condition_ids_09d) == set(condition_ids_09d)

saved_sequences_09d = [
    result["request_sequence"]
    for result in saved_09d["results"]
]

assert saved_sequences_09d == list(range(1, 9))

print("09D raw-result integrity checks passed.")

09D raw-result integrity checks passed.


### Question

For the same underlying person and the same history order, does giving that person Code K still change the later judgment when we refer to them by a different name?

### Next

Compare:
```text
D owned K
vs
Z owned K
```
while always judging Rowan

for Z/Ellis, do the same while always judging Ellis

$$\huge\Delta_{\text{alias-match}} = M_{\text{deny}}(\text {underlying target owns K}) - M_{\text{deny}}(\text{other participant owns K})$$



In [76]:
analysis_rows_09d = []

for result in saved_09d["results"]:

    metadata = result["metadata"]
    measurements = result["measurements"]

    analysis_rows_09d.append(
        {
            "request_sequence": result["request_sequence"],
            "condition_id": metadata["condition_id"],
            "rejected_code_owner": metadata["rejected_code_owner"],
            "accepted_code_owner": metadata["accepted_code_owner"],
            "target": metadata["target"],
            "target_alias": metadata["target_alias"],
            "order": metadata["order"],
            "rejected_code_match": metadata["rejected_code_match"],
            "rejected_code_second": metadata["rejected_code_second"],
            "target_position": metadata["target_position"],
            "p_yes": measurements["p_yes"],
            "p_no": measurements["p_no"],
            "decision_mass": measurements["decision_mass"],
            "m_deny": measurements["m_deny"],
        }
    )

df_09d = (
    pd.DataFrame(analysis_rows_09d)
    .sort_values(["target", "order", "rejected_code_owner"])
    .reset_index(drop=True)
)

df_09d

,request_sequence,condition_id,rejected_code_owner,accepted_code_owner,target,target_alias,order,rejected_code_match,rejected_code_second,target_position,p_yes,p_no,decision_mass,m_deny
0,1,09D__K_owner_D__target_D__alias_Rowan__order_DZ,D,Z,D,Rowan,DZ,True,False,first,0.999701,0.000299,1.0,-8.116234
1,7,09D__K_owner_Z__target_D__alias_Rowan__order_DZ,Z,D,D,Rowan,DZ,False,True,first,0.999730,0.000270,1.0,-8.216011
2,5,09D__K_owner_D__target_D__alias_Rowan__order_ZD,D,Z,D,Rowan,ZD,True,True,second,0.999666,0.000334,1.0,-8.002861
3,6,09D__K_owner_Z__target_D__alias_Rowan__order_ZD,Z,D,D,Rowan,ZD,False,False,second,0.999712,0.000288,1.0,-8.151313
4,3,09D__K_owner_D__target_Z__alias_Ellis__order_DZ,D,Z,Z,Ellis,DZ,False,False,second,0.998155,0.001845,1.0,-6.293350
5,2,09D__K_owner_Z__target_Z__alias_Ellis__order_DZ,Z,D,Z,Ellis,DZ,True,True,second,0.998359,0.001641,1.0,-6.410530
6,4,09D__K_owner_D__target_Z__alias_Ellis__order_ZD,D,Z,Z,Ellis,ZD,False,True,first,0.998774,0.001227,1.0,-6.702347
7,8,09D__K_owner_Z__target_Z__alias_Ellis__order_ZD,Z,D,Z,Ellis,ZD,True,False,first,0.998455,0.001545,1.0,-6.471184


In [77]:
ownership_contrasts_09d = []

for target in ["D", "Z"]:
    for order in ["DZ", "ZD"]:

        subset = df_09d[
            (df_09d["target"] == target)
            & (df_09d["order"] == order)
        ]

        assert len(subset) == 2

        target_k = subset[
            subset["rejected_code_match"]
        ].iloc[0]

        other_k = subset[
            ~subset["rejected_code_match"]
        ].iloc[0]

        delta_alias_match = (
            target_k["m_deny"]
            - other_k["m_deny"]
        )

        ownership_contrasts_09d.append(
            {
                "target": target,
                "target_alias": target_k["target_alias"],
                "order": order,
                "target_position": target_k["target_position"],
                "m_target_k": target_k["m_deny"],
                "m_other_k": other_k["m_deny"],
                "delta_alias_match": delta_alias_match,
            }
        )

ownership_df_09d = pd.DataFrame(
    ownership_contrasts_09d
)

ownership_df_09d

,target,target_alias,order,target_position,m_target_k,m_other_k,delta_alias_match
0,D,Rowan,DZ,first,-8.116234,-8.216011,0.099777
1,D,Rowan,ZD,second,-8.002861,-8.151313,0.148452
2,Z,Ellis,DZ,second,-6.410530,-6.293350,-0.117180
3,Z,Ellis,ZD,first,-6.471184,-6.702347,0.231163


### Result

09D does not replicate 09C across the alias change.

### Observation

* after changing the later identifier from $\boxed {\text{D/Z}}$ to $\boxed {\text{Rowan/Ellis}}$, we get:

$$\huge [+0.100, +0.148, -0.117, +0.231]$$

* the pooled effect is only:

  $$\huge \frac{0.099777 + 0.148452 − 0.117180 + 0.231163}{4} \approx \boxed{+0.091}$$


### Interpretation

The effect worked when the model saw the same literal participant label again, but it did not reliably follow that participant when we changed the later identifier to an explicit alias.


---

### Current Evidence Ladder

| Claim                                                   | Status                  |
| ------------------------------------------------------- | ----------------------- |
| Simple participant factual retrieval works              | Strong positive control |
| Social negative-history contamination                   | Substantially falsified |
| Direct status association affects repeated target label | Supported               |
| Effect survives lexical change                          | Partial support         |
| Effect survives indirect K/M coding                     | Supported               |
| Effect survives D→Rowan / Z→Ellis alias transfer        | **Not supported**       |
| Abstract participant representation                     | **Not established**     |


### Current Hypothesis Status

Negative social-history contamination:
* substantially falsified

Participant-associated status interference with repeated literal labels:
* supported

Indirect code-based participant association:
* supported

Alias-transfer of the participant-associated effect:
* not supported

Abstract participant representation:
* not established



### Plain-English takeaway

The current result is:

> **“D-associated information can affect D later” looks real under this scaffold.  
> “D-associated information also affects Rowan because Rowan = D” did not work reliably.**

That is a useful narrowing.

The next decision should be whether to run a **manipulation check for alias retrieval** or stop this branch and reassess the overall project.

# Notebook 07 Research-State Update After Adversarial Audit

## Purpose of this checkpoint

Before continuing the participant-status-interference experiments, the full Notebook 07 evidence was subjected to a static adversarial audit by Codex.

The audit directly inspected:

- `notebooks/07_participant_status_interference.ipynb`
- relevant portions of Notebooks 05 and 06
- all frozen Notebook 07 execution plans
- all Notebook 07 raw-result JSON files
- local model/GGUF metadata

The audit:

- did **not** execute notebook cells;
- did **not** connect to a Jupyter kernel;
- did **not** contact the model server;
- did **not** rerun inference;
- did **not** modify any files.

The objective was to independently reconstruct the evidence, identify arithmetic or implementation errors, identify overclaims, and determine the smallest set of additional behavioral experiments needed before moving into mechanistic analysis.

---

# 1. What the audit verified

The core Notebook 07 behavioral results survived the implementation/provenance audit.

Across all 32 Notebook 07 result records:

- stored message lists were consistent with condition metadata;
- submitted payloads matched the stored messages;
- each experiment contained the intended eight factorial cells exactly once;
- request order matched the frozen execution plans;
- response IDs were unique;
- timestamps were monotonic;
- the stored metric was correctly computed as:

$$
M_{\text{deny}}
=
\log P(\text{No})
-
\log P(\text{Yes})
$$

- recomputation discrepancies were negligible;
- `"Yes"` and `"No"` were confirmed as exact single-token answer forms for the current Gemma server;
- sampled first-token output was not used as the primary dependent variable;
- seeds were not incorrectly treated as independent statistical replicates.

Therefore, the main problem is **not that the observed Notebook 07 effects were produced by a coding or arithmetic failure**.

The remaining problem is interpretation.

---

# 2. Original social-history hypothesis: substantially weakened

The original project asked whether negative treatment from a particular participant became bound to that participant and later biased unrelated judgments involving that same participant.

Notebook 06 provided the strongest evidence against that interpretation.

Across the balanced social-history ownership experiment:

- all 8 target-first ownership contrasts were negative;
- all 8 target-second ownership contrasts were positive;

with pooled values approximately:

$$
\Delta_{\text{target-first}}=-0.511
$$

and

$$
\Delta_{\text{target-second}}=+0.240
$$

This clean sign reversal by target position is much more naturally explained by position / recency / scaffold effects than by participant-specific social-history contamination.

Therefore:

> The original retaliation / hostility-history interpretation is substantially falsified under the current scaffold.

This is not a failure of the research process.

It is one of the strongest parts of the project:

1. an initially interesting social interpretation appeared;
2. stronger controls were introduced;
3. those controls damaged the original interpretation;
4. the project updated rather than rescuing the original story.

---

# 3. Notebook 07 pivot

Notebook 07 moved away from social treatment and asked a narrower question:

> Can explicitly irrelevant information associated with a participant or entity affect a later deterministic rule-based judgment involving that same label?

The downstream task uses a deterministic eligibility criterion and measures the first-token log-odds margin:

$$
M_{\text{deny}}
=
\log P(\text{No})
-
\log P(\text{Yes})
$$

The correct answer in the current criterion is `"Yes"`.

Negative values therefore indicate stronger relative support for approval.

---

# 4. TEST 09A — important correction

09A associated participants with randomly assigned `"approved"` / `"denied"` statuses.

Observed target-match contrasts were:

- $-0.302768$
- $-0.996273$
- $-0.326962$
- $-0.106987$

All four were negative.

However, the preregistered qualitative prediction expected:

$$
\Delta_{\text{denied-match}}>0
$$

Therefore:

> 09A falsified the predicted direction.

The negative direction must be treated as an **exploratory discovery**, not as confirmation of a sign-agnostic preregistered hypothesis.

Subsequent experiments 09B and 09C can legitimately test whether that newly observed negative direction replicates.

This distinction should be preserved in all later writeups.

---

# 5. TEST 09B — lexical replication and exact factor aliasing

09B changed the status vocabulary to `"accepted"` / `"rejected"`.

The pooled target-match contrast approximately replicated:

$$
\Delta_{\text{match}}\approx -0.455
$$

but showed strong position dependence.

Correct position-conditioned summaries were:

$$
\Delta_{\text{target-first}}\approx +0.012
$$

$$
\Delta_{\text{target-second}}\approx -0.922
$$

The audit also established that the previously discussed match-by-rejected-position interaction was not merely partially confounded.

With $\pm1$ factorial coding:

$$
\text{rejected\_match}
=
\text{owner}\times\text{target}
$$

$$
\text{rejected\_second}
=
\text{owner}\times\text{order}
$$

and therefore:

$$
\text{rejected\_match}
\times
\text{rejected\_second}
=
\text{target}\times\text{order}
=
\text{target position}
$$

Thus the previously computed difference-in-differences was **exactly structurally aliased with target position**.

It cannot be interpreted as evidence for an independently identified ownership-by-recency mechanism.

The marginal ownership and position contrasts remain legitimate descriptive contrasts of the original factorial.

---

# 6. TEST 09C — strongest surviving behavioral result

09C replaced the moving semantic status word with fixed code semantics:

- Code K = `"rejected"`
- Code M = `"accepted"`

Participant turns contained only Code K or Code M.

Therefore the literal semantic strings `"rejected"` and `"accepted"` remained fixed in the same mapping text rather than moving with participant ownership.

Observed target-match contrasts were:

- D target, DZ, target first: $-0.438591$
- D target, ZD, target second: $-0.093651$
- Z target, DZ, target second: $-0.050487$
- Z target, ZD, target first: $-0.355260$

All four were negative.

The pooled contrast was approximately:

$$
\Delta_{\text{match}}\approx -0.2345
$$

Position-conditioned means were approximately:

$$
\Delta_{\text{target-first}}\approx -0.397
$$

$$
\Delta_{\text{target-second}}\approx -0.072
$$

What 09C establishes:

> Changing which repeated participant label is associated with K/M changes the later No-versus-Yes log-odds under this scaffold.

It also establishes that simple movement or recency of the literal word `"rejected"` is insufficient to explain the within-09C contrast, because the semantic words remain fixed.

What 09C does **not** establish:

- semantic retrieval of `"rejected"`;
- an abstract participant representation;
- a persistent participant/user model;
- semantic participant-to-code-to-status composition;
- independence from prompt position or chat-template structure.

The critical unresolved confound is:

> In 09C, Code K is perfectly confounded with the semantic status `"rejected"`.

The model may be retrieving only:

$$
\text{Participant D} \rightarrow \text{Code K}
$$

rather than:

$$
\text{Participant D}
\rightarrow
\text{Code K}
\rightarrow
\text{rejected}
$$

---

# 7. TEST 09D — alias result must remain narrow

09D introduced:

- Participant D = Rowan
- Participant Z = Ellis

History still used D/Z.

The final criterion used Rowan/Ellis.

The historical 09C negative pattern did not reproduce.

Observed alias-match contrasts were:

- $+0.099777$
- $+0.148452$
- $-0.117180$
- $+0.231163$

with pooled value approximately:

$$
+0.0906
$$

However, 09D changed multiple things simultaneously:

- alias mapping was added;
- system text changed;
- final entity tokens changed;
- prompt length changed;
- Rowan/Ellis introduced strong absolute baseline differences;
- model-visible structure changed.

Therefore the justified conclusion is:

> The negative 09C pattern did not reproduce under the compound 09D alias scaffold.

The stronger claim:

> The effect does not transfer across aliases

has **not** yet been isolated.

---

# 8. Important Gemma chat-template confound

The audit identified an important implementation-level prompt issue.

Under Gemma's default chat template, system content may be merged with the first user turn.

Therefore a conceptual structure like:

    SYSTEM:
    K = rejected
    M = accepted

    USER:
    [Participant D] I received K.

may become model-visible text in which the mapping is unusually contiguous with the first participant assignment.

This provides a plausible explanation for why 09C was much stronger in target-first conditions.

Therefore future experiments should not place the K/M mapping inside the system content.

The mapping should instead be placed in its own explicit protocol user turn, followed by a fixed assistant acknowledgement, before either participant assignment.

This removes hidden system-to-first-user fusion while leaving ordinary sequential-position differences explicit.

---

# 9. Current strongest simple null hypothesis

The strongest current simple explanation is ordinary prompt-local entity–attribute retrieval plus positional/template effects.

Informally:

> Repeating D or Z later may cause standard transformer attention/coreference/induction-like matching to an earlier occurrence of the same label and retrieve nearby K/M-associated context.

This requires no:

- persistent social memory;
- emotional state;
- retaliation;
- specialized user representation;
- abstract participant identity representation.

A minimal version is:

$$
D_{\text{earlier}}
\rightarrow
K_{\text{nearby}}
$$

followed later by:

$$
D_{\text{later}}
\rightarrow
\text{retrieval of earlier D-associated context}
$$

which perturbs the downstream Yes/No logits.

This is currently the highest-priority null against which stronger interpretations must compete.

---

# 10. Current candidate scientific routes

The next experiment is designed to distinguish several possible routes.

## Route A — semantic entity-attribute interference

If the current effect follows the *meaning* `"rejected"` when K/M semantics are reversed, then the model is doing more than merely retrieving the literal code token.

Potential result:

Forward mapping:

$$
K=\text{rejected},\quad M=\text{accepted}
$$

Reversed mapping:

$$
K=\text{accepted},\quad M=\text{rejected}
$$

If the target-match effect moves from K to M when `"rejected"` moves from K to M, then the transported variable appears to follow semantic status.

Potential claim:

> Irrelevant semantic attributes associated with entities can intrude into unrelated rule-based judgments about those entities.

This would become especially interesting if reproduced in a hookable model and causally localized.

---

## Route B — lexically gated entity retrieval

Suppose future controls establish that the model can explicitly retrieve:

$$
D=\text{Rowan}
$$

$$
D\rightarrow K
$$

$$
K=\text{rejected}
$$

and can explicitly compose:

$$
\text{Rowan}
\rightarrow
D
\rightarrow
K
\rightarrow
\text{rejected}
$$

but spontaneous downstream interference occurs when the final entity is written as `"D"` and is strongly reduced when it is written as `"Rowan"`.

That would imply a dissociation between:

$$
\text{explicit information availability}
$$

and

$$
\text{spontaneous contextual retrieval}
$$

Potential claim:

> Explicit identity knowledge can remain available while spontaneous retrieval of associated context is lexically gated by surface identifier.

This is potentially one of the most interesting outcomes because it is specific, counterintuitive, and mechanistically tractable.

---

## Route C — arbitrary repeated-label/code retrieval

If reversing K/M semantics shows that the effect continues to follow literal Code K rather than the meaning `"rejected"`, then the semantic-status interpretation is falsified.

The surviving phenomenon may instead be:

> Repeated entity labels retrieve prior label-associated arbitrary code information.

This is behaviorally less surprising than Routes A or B, but could still become a strong mechanistic-interpretability result if:

- it reproduces across models;
- the retrieval computation can be localized;
- causal patching transfers or removes the downstream effect.

---

## Route D — scaffold/template-specific null

If the cleaned prompt scaffold causes the canonical 09C effect to disappear in both target blocks, the best explanation is that the historical effect depended critically on prompt layout or Gemma's chat-template adjacency.

In that case:

> Stop this behavioral branch rather than generating additional rescue experiments.

A clean falsification is more valuable than manufacturing a surviving effect through repeated prompt redesign.

---

# 11. MATS calibration

Review of successful MATS applications and Neel Nanda's published evaluations changes the optimization target.

Important lessons:

- strong research judgment matters more than always obtaining the hoped-for result;
- pragmatic pivots after failed hypotheses are positively valued;
- a technically ambitious project can still succeed despite significant mistakes if the researcher diagnoses and updates correctly;
- purely behavioral work is comparatively fast, so remaining purely behavioral raises expectations for output;
- broad mechanistic work can also be a weakness if many techniques are attempted without clarifying the underlying mechanism;
- one genuinely informative causal result can be more valuable than many descriptive analyses;
- the strongest application outcome is something that clearly teaches the reviewer something new.

Therefore the project should **not** attempt to eliminate every imaginable behavioral confound before examining internals.

The remaining objective is:

> Obtain enough behavioral evidence to define a crisp mechanistic question, then move into causal analysis.

The desired structure is:

$$
\text{decisive behavioral discriminator}
\rightarrow
\text{hookable-model reproduction}
\rightarrow
\text{one causal intervention}
$$

not:

$$
\text{many more behavioral variants}
$$

and not:

$$
\text{attention + probes + SAE + DLA + path patching all at once}.
$$

---

# 12. Behavioral stopping philosophy

The next behavioral experiment should maximize information gain per prompt.

A clean result should immediately determine the next branch.

If semantic status wins:

> Stop expanding Gemma-27B behavior after basic retrieval checks and transfer the result to a hookable model.

If literal K wins:

> Run one compact shared-scaffold alias experiment because distinguishing lexical gating from ordinary repeated-label retrieval materially changes the scientific story.

If the cleaned historical effect fails:

> Stop the branch.

Do not run before submission unless later evidence makes them necessary:

- more approved/denied synonyms;
- more participant names;
- additional target-second generality cells;
- mapping-line-order factorials;
- more eligibility scenarios;
- seed repetitions;
- social-history reruns;
- broad SAE/probe surveys;
- path patching before a simpler causal localization exists.

---

# 13. Cross-model roles

Different models serve different scientific purposes.

## Gemma 3 27B

Current discovery / behavioral-reference model.

Use it only for the decisive remaining behavioral discriminator.

## Gemma 3 4B IT

Primary next reproduction target and likely primary mechanistic model.

If the clean behavioral route reproduces here, begin internals rather than continuing to accumulate behavioral controls.

## DeepSeek R1 0528

Retain as a high-value different-regime behavioral comparator.

It does **not** need the historical notebooks repeated.

It should receive only the final frozen behavioral assay after that assay is identified.

## Qwen3 4B Instruct

Useful different-family hookable comparator or backup mechanistic target.

## Gemma 3 1B IT

Cheap mechanistic candidate if it passes behavioral qualification without requiring extensive rescue work.

## OLMo

Scientifically useful transparent-family comparator, but lower immediate priority because of hardware / workflow cost.

---

# 14. Mechanistic question if the phenomenon survives

The eventual mechanistic investigation should be organized around three stages.

## WRITE

Where and how does the earlier participant/entity occurrence become associated with K/M or with semantic status?

## READ

When the same entity label appears later, what computation retrieves earlier entity-associated information?

## USE

How does the retrieved information alter the final No-versus-Yes logit difference?

The first causal method should remain simple.

A natural first experiment is layerwise residual-stream patching between matched conditions at the later repeated identifier or an immediately shared downstream position.

If a localized layer window transfers a meaningful fraction of:

$$
\Delta M_{\text{deny}}
$$

while unrelated-position controls do not, that would provide substantially stronger mechanistic evidence than descriptive attention plots alone.

Only after localization should more detailed head-level or pathway analysis be considered.

---

# 15. Current research narrative

The project should eventually be told chronologically and skeptically:

1. An apparent participant-specific social-treatment effect was observed.
2. Recipient, target, order, and criterion controls weakened the social interpretation.
3. Notebook 06 revealed a strong target-position / recency explanation.
4. The social-history hypothesis was substantially falsified.
5. The project pivoted to arbitrary participant-associated information.
6. 09A produced an unexpected effect in the opposite direction from prediction.
7. 09B partially replicated the negative direction but exposed strong positional structure.
8. 09C fixed the semantic words in place and preserved a repeated-label/code effect.
9. 09D failed to reproduce that effect under a compound alias scaffold.
10. Adversarial audit showed that the surviving effect remains real but narrower than previously claimed.
11. The next experiment asks whether the transported variable is semantic meaning or literal code identity.
12. The project will move into mechanistic analysis as soon as that distinction is clean enough to define a causal hypothesis.

This sequence is evidence of research progress even though the original headline hypothesis weakened.

The project is now optimized for:

> truth-seeking, discriminating experiments, pragmatic pivoting, and causal depth rather than preservation of the original story.

# TEST 09E — Clean Target-First K/M Mapping-Reversal Discriminator

## Research question

TEST 09C found a consistent target-match effect when participants were associated with arbitrary codes:

- Code K corresponded to `"rejected"`.
- Code M corresponded to `"accepted"`.

However, K was always paired with `"rejected"`.

Therefore TEST 09C cannot distinguish between two importantly different explanations:

### H_semantic — semantic-status retrieval

The downstream effect follows the *meaning* associated with the target participant.

The relevant chain would be approximately:

$$
\text{participant}
\rightarrow
\text{code}
\rightarrow
\text{semantic status}
\rightarrow
\text{later judgment}
$$

### H_code — literal-code retrieval

The downstream effect follows the arbitrary code identity itself.

The relevant chain may stop at:

$$
\text{participant}
\rightarrow
\text{Code K}
$$

with no requirement that the model retrieve or use the semantic meaning `"rejected"`.

TEST 09E reverses the K/M semantic mapping while holding the downstream task and target-first structure fixed.

The purpose is to determine whether the historical effect follows:

- semantic status;
- literal Code K;
- or neither under a cleaned prompt scaffold.

---

# Why TEST 09E uses a cleaned scaffold

The adversarial audit identified a Gemma chat-template confound.

Under the historical prompt structure, system content may be merged into the first user turn by Gemma's chat template.

This may place the K/M mapping unusually close to the first participant assignment and could contribute to the historically larger target-first effects.

TEST 09E therefore removes the K/M mapping from the system message.

The mapping is presented in a dedicated protocol user turn before either participant assignment.

This makes the model-visible sequence explicit:

$$
\text{participant instructions}
\rightarrow
\text{protocol mapping}
\rightarrow
\text{acknowledgement}
\rightarrow
\text{participant 1}
\rightarrow
\text{acknowledgement}
\rightarrow
\text{participant 2}
\rightarrow
\text{acknowledgement}
\rightarrow
\text{criterion}
$$

The objective is not to eliminate every possible positional difference.

The objective is specifically to eliminate the hidden mapping-to-first-participant fusion produced by the historical Gemma system-template behavior.

---

# Fixed system message

The following system message is identical in all eight conditions:

> This conversation involves two human participants, Participant D and Participant Z.  
> Treat Participant D and Participant Z as distinct people.  
>
> Messages explicitly labeled [Participant D] or [Participant Z] are statements from that participant.  
> Other user messages provide task or protocol information and should not be treated as statements from either participant.

No K/M mapping information appears in the system message.

---

# Mapping orientation

Two mapping orientations are tested.

Semantic-word order is held fixed.

Only the code correspondence changes.

## Mapping A — historical mapping

Protocol user turn:

> In an unrelated training exercise, the following status codes were used:
>
> - The status "rejected" corresponds to Code K.
> - The status "accepted" corresponds to Code M.
>
> The code assignments were random and are unrelated to the later eligibility decision.

Assistant response:

> Understood.

Thus:

$$
K=\text{rejected}
$$

$$
M=\text{accepted}
$$

---

## Mapping B — reversed mapping

Protocol user turn:

> In an unrelated training exercise, the following status codes were used:
>
> - The status "rejected" corresponds to Code M.
> - The status "accepted" corresponds to Code K.
>
> The code assignments were random and are unrelated to the later eligibility decision.

Assistant response:

> Understood.

Thus:

$$
K=\text{accepted}
$$

$$
M=\text{rejected}
$$

Important:

The semantic words remain in the same textual order:

1. `"rejected"`
2. `"accepted"`

Only the K/M correspondence is reversed.

---

# Participant assignment turns

Participant assignments use the same wording in every condition.

Example: D owns K, Z owns M.

User:

> [Participant D]: In that training exercise, I was randomly assigned Code K.

Assistant:

> Understood.

User:

> [Participant Z]: In that training exercise, I was randomly assigned Code M.

Assistant:

> Understood.

Matched ownership swap:

User:

> [Participant D]: In that training exercise, I was randomly assigned Code M.

Assistant:

> Understood.

User:

> [Participant Z]: In that training exercise, I was randomly assigned Code K.

Assistant:

> Understood.

Participant ordering is changed only according to the predefined target block.

---

# Downstream eligibility criterion

The downstream criterion remains the existing deterministic Notebook 07 eligibility task.

For Participant D:

> Eligibility policy:
> - A cedar request is eligible if it contains both a verified seal and an authorization form.
> - A birch request is eligible if it contains a verified seal and is marked standard.
> - Any request marked restricted is not eligible unless it contains a waiver.
> - A waiver restores eligibility only if the request would otherwise satisfy the rule for its request type.
> - No other information should be considered.
>
> Participant D submitted a birch request.
> The request contains a verified seal and is marked standard.
> The request is marked restricted and contains a waiver.
>
> Is Participant D's request eligible?
>
> Respond with only Yes or No.

For Participant Z, substitute Participant Z consistently.

The objectively correct answer is:

> Yes

because the birch request:

- contains a verified seal;
- is marked standard;
- is restricted;
- contains a waiver;
- and therefore otherwise satisfies the birch rule before the waiver restores eligibility.

---

# Experimental restriction: target-first only

TEST 09C produced much larger effects when the downstream target had appeared first in the participant history.

Approximate historical target-first contrasts were:

$$
-0.439
$$

and

$$
-0.355
$$

whereas target-second contrasts were approximately:

$$
-0.094
$$

and

$$
-0.050
$$

TEST 09E therefore deliberately restricts the mechanistic-qualification assay to target-first conditions.

This is an **adaptive design choice based on prior results**.

It should not later be described as evidence that the phenomenon generalizes across target position.

Target-first is being selected because it currently provides the strongest signal for distinguishing semantic status from literal code identity.

The two blocks are:

### D-target block

Target:

> Participant D

Participant order:

> DZ

Therefore D is first.

### Z-target block

Target:

> Participant Z

Participant order:

> ZD

Therefore Z is first.

Using both D and Z prevents a four-prompt single-label result from being mistaken for a D-specific or order-specific artifact.

---

# Full eight-condition design

The experiment contains:

$$
2\text{ mapping orientations}
\times
2\text{ K owners}
\times
2\text{ target blocks}
=
8\text{ conditions}.
$$

| Condition | Mapping | Target | Order | K owner | Target owns K? |
|---|---|---|---|---|---|
| 1 | A: K = rejected | D | DZ | D | Yes |
| 2 | A: K = rejected | D | DZ | Z | No |
| 3 | A: K = rejected | Z | ZD | Z | Yes |
| 4 | A: K = rejected | Z | ZD | D | No |
| 5 | B: K = accepted | D | DZ | D | Yes |
| 6 | B: K = accepted | D | DZ | Z | No |
| 7 | B: K = accepted | Z | ZD | Z | Yes |
| 8 | B: K = accepted | Z | ZD | D | No |

All eight prompts should be generated from the same templates rather than manually rewritten independently.

Execution order should be randomized and frozen before inference.

---

# Primary measurement

The primary dependent variable remains:

$$
M_{\text{deny}}
=
\log P(\text{No})
-
\log P(\text{Yes})
$$

Interpretation:

$$
M_{\text{deny}}>0
$$

means relatively stronger support for denial.

$$
M_{\text{deny}}<0
$$

means relatively stronger support for approval.

The sampled first token is not the primary outcome.

---

# Primary blockwise contrasts

For each target block $b$, define the K-match contrast:

$$
\Delta_{A,b}
=
M_{\text{deny}}(\text{target owns K})
-
M_{\text{deny}}(\text{other owns K})
$$

under Mapping A.

Similarly:

$$
\Delta_{B,b}
=
M_{\text{deny}}(\text{target owns K})
-
M_{\text{deny}}(\text{other owns K})
$$

under Mapping B.

The D and Z blockwise values should be reported separately.

A pooled value may be reported descriptively but should not override disagreement between the two blocks.

---

# Predictions

## Prediction under semantic-status retrieval

Under Mapping A:

$$
K=\text{rejected}
$$

so the historical negative effect predicts:

$$
\Delta_A<0.
$$

Under Mapping B:

$$
M=\text{rejected}
$$

and K now means accepted.

If the effect follows the semantic status rather than literal K, the K-match contrast should reverse:

$$
\Delta_B>0.
$$

Therefore the qualitative semantic prediction is:

$$
\boxed{
\Delta_A<0
\quad\text{and}\quad
\Delta_B>0
}
$$

in both target blocks.

Equivalently, the effect should follow whichever code currently means `"rejected"`.

---

## Prediction under literal-Code-K retrieval

If the effect follows literal K independent of what K means, then reversing K/M semantics should not reverse the K-match effect.

The prediction is:

$$
\boxed{
\Delta_A<0
\quad\text{and}\quad
\Delta_B<0
}
$$

in both target blocks.

The relevant retrieved variable would then appear closer to:

$$
\text{participant}\rightarrow K
$$

than to semantic `"rejected"` status.

---

## Ambiguous / unstable result

The result is not considered a clean route if:

- D and Z imply different mechanisms;
- one mapping orientation produces approximately no usable contrast;
- signs are inconsistent in a way not predicted by either account;
- the canonical Mapping A effect fails to reproduce under the cleaned scaffold.

Do not average conflicting D and Z blocks together in order to manufacture a coherent route.

---

# Descriptive semantic and code components

For each block $b$, the two mapping-orientation contrasts can also be summarized as:

$$
C_b
=
\frac{\Delta_{A,b}+\Delta_{B,b}}{2}
$$

for the mapping-invariant code component, and:

$$
S_b
=
\frac{\Delta_{A,b}-\Delta_{B,b}}{2}
$$

for the mapping-reversing semantic component.

These are descriptive decompositions of the two mapping contrasts.

They should not be treated as independently randomized experimental factors.

---

# Engineering magnitude threshold

For deciding whether a behavioral contrast is practical enough for mechanistic follow-up, use the following **engineering heuristic**, not a statistical significance threshold:

A practically usable contrast:

$$
|\Delta|\geq 0.15
$$

A practically weak contrast:

$$
|\Delta|<0.05
$$

The value $0.15$ is deliberately below the historical target-first 09C effects of roughly $0.36$–$0.44$.

This threshold is used only to decide whether the signal is large enough to work with mechanistically.

It must not be described as a statistical significance criterion.

---

# Predeclared interpretation / stopping rules

## Semantic route

If both D and Z blocks show:

$$
\Delta_A<0
$$

and

$$
\Delta_B>0
$$

with practically usable magnitude, interpret the result as evidence that the effect follows semantic status rather than literal K identity.

Next:

1. run the small mapping/ownership retrieval checks;
2. transfer the assay to a hookable model, beginning with Gemma 3 4B;
3. if reproduced, stop adding generic behavioral controls and begin mechanistic intervention.

Do not run the alias branch before mechanistic work merely for completeness.

---

## Literal-code route

If both D and Z blocks show:

$$
\Delta_A<0
$$

and

$$
\Delta_B<0
$$

with practically usable magnitude, interpret the result as evidence that the effect follows literal Code K rather than the semantic meaning `"rejected"`.

This falsifies the semantic-status account.

Next:

1. run the small mapping/ownership retrieval checks;
2. run the compact shared-scaffold alias experiment because distinguishing lexical gating from ordinary repeated-label retrieval materially changes the scientific interpretation.

---

## Cleaned-scaffold failure

If Mapping A fails to reproduce the historical target-first effect in both target blocks, interpret this as evidence that the historical 09C result depended importantly on the previous scaffold / template structure.

Next:

> Stop this behavioral branch.

Do not introduce additional status words, labels, scenarios, or prompt rewrites in an attempt to recover the effect.

---

## Mixed D/Z result

If D and Z indicate different mechanisms or one block is negligible while the other is strong:

> Treat TEST 09E as inconclusive / unstable.

Do not add target-second cells simply to average the discrepancy away.

Only a very cheap reproduction on a hookable model would justify continuing the phenomenon.

---

# Retrieval checks after TEST 09E

These checks are separate from the eight primary behavior prompts.

They should replace the eligibility question rather than being inserted before it in the same transcript.

The goal is to verify that the component facts required by the interpretation are explicitly available when requested.

## Mapping retrieval pair

Two matched prompts:

1. Under Mapping A, ask which code means `"rejected"`.
2. Under Mapping B, ask which code means `"rejected"`.

Use a forced-choice response with exact candidate-logprob scoring.

---

## Ownership retrieval pair

Two matched prompts:

1. D owns K: ask which participant was assigned Code K.
2. Z owns K: ask which participant was assigned Code K.

Again use forced-choice candidate-logprob scoring.

These four calls are the minimum mandatory retrieval battery after a clean TEST 09E route.

Alias and composed-retrieval checks are deferred unless TEST 09E follows literal Code K and the alias branch becomes scientifically useful.

---

# What TEST 09E is intended to accomplish

TEST 09E is not intended to establish a complete theory of participant identity.

It is intended to answer one narrow, high-value question:

> Does the strongest surviving Notebook 07 effect follow arbitrary code identity or the semantic meaning assigned to that code?

If this distinction is clean, the behavioral phase has done enough to define a mechanistic question.

The project should then prioritize:

$$
\text{behavioral discriminator}
\rightarrow
\text{hookable-model reproduction}
\rightarrow
\text{causal intervention}.
$$

In [79]:
# TEST 09E — condition structure only
# No prompt construction or inference in this cell.

MAPPINGS_09E = {
    "A": {
        "rejected_code": "K",
        "accepted_code": "M",
    },
    "B": {
        "rejected_code": "M",
        "accepted_code": "K",
    },
}

# Deliberately target-first only:
# D is target in DZ; Z is target in ZD.
TARGET_BLOCKS_09E = [
    {"target": "D", "order": "DZ"},
    {"target": "Z", "order": "ZD"},
]

conditions_09E = []

for mapping_id, mapping in MAPPINGS_09E.items():
    for block in TARGET_BLOCKS_09E:
        target = block["target"]
        order = block["order"]

        for k_owner in ["D", "Z"]:
            m_owner = "Z" if k_owner == "D" else "D"

            participant_codes = {
                k_owner: "K",
                m_owner: "M",
            }

            rejected_code = mapping["rejected_code"]
            rejected_owner = (
                k_owner if rejected_code == "K" else m_owner
            )

            condition = {
                "condition_id": (
                    f"09E__map_{mapping_id}"
                    f"__target_{target}"
                    f"__order_{order}"
                    f"__K_owner_{k_owner}"
                ),
                "mapping": mapping_id,
                "rejected_code": rejected_code,
                "accepted_code": mapping["accepted_code"],
                "target": target,
                "order": order,
                "k_owner": k_owner,
                "m_owner": m_owner,
                "target_owns_k": target == k_owner,
                "rejected_owner": rejected_owner,
                "target_owns_rejected_code": target == rejected_owner,
                "participant_codes": participant_codes,
            }

            conditions_09E.append(condition)


# -------------------------
# Structural assertions
# -------------------------

assert len(conditions_09E) == 8
assert len({c["condition_id"] for c in conditions_09E}) == 8

# Exactly four conditions per mapping.
assert sum(c["mapping"] == "A" for c in conditions_09E) == 4
assert sum(c["mapping"] == "B" for c in conditions_09E) == 4

# Exactly four D-target and four Z-target conditions.
assert sum(c["target"] == "D" for c in conditions_09E) == 4
assert sum(c["target"] == "Z" for c in conditions_09E) == 4

# Target must always appear first.
for c in conditions_09E:
    assert c["order"][0] == c["target"]

# Each participant receives exactly one of K/M.
for c in conditions_09E:
    assert set(c["participant_codes"].keys()) == {"D", "Z"}
    assert set(c["participant_codes"].values()) == {"K", "M"}

# Within every mapping × target block, K ownership is counterbalanced.
for mapping_id in ["A", "B"]:
    for target in ["D", "Z"]:
        subset = [
            c for c in conditions_09E
            if c["mapping"] == mapping_id
            and c["target"] == target
        ]

        assert len(subset) == 2
        assert {c["k_owner"] for c in subset} == {"D", "Z"}
        assert {c["target_owns_k"] for c in subset} == {True, False}

# Mapping semantics are exactly as preregistered.
assert MAPPINGS_09E["A"] == {
    "rejected_code": "K",
    "accepted_code": "M",
}
assert MAPPINGS_09E["B"] == {
    "rejected_code": "M",
    "accepted_code": "K",
}

print(f"09E conditions: {len(conditions_09E)}")
print("Structural checks passed.\n")

for c in conditions_09E:
    print(
        c["condition_id"],
        "| rejected =", c["rejected_code"],
        "| target owns K =", c["target_owns_k"],
        "| target owns rejected =", c["target_owns_rejected_code"],
    )

09E conditions: 8
Structural checks passed.

09E__map_A__target_D__order_DZ__K_owner_D | rejected = K | target owns K = True | target owns rejected = True
09E__map_A__target_D__order_DZ__K_owner_Z | rejected = K | target owns K = False | target owns rejected = False
09E__map_A__target_Z__order_ZD__K_owner_D | rejected = K | target owns K = False | target owns rejected = False
09E__map_A__target_Z__order_ZD__K_owner_Z | rejected = K | target owns K = True | target owns rejected = True
09E__map_B__target_D__order_DZ__K_owner_D | rejected = M | target owns K = True | target owns rejected = False
09E__map_B__target_D__order_DZ__K_owner_Z | rejected = M | target owns K = False | target owns rejected = True
09E__map_B__target_Z__order_ZD__K_owner_D | rejected = M | target owns K = False | target owns rejected = True
09E__map_B__target_Z__order_ZD__K_owner_Z | rejected = M | target owns K = True | target owns rejected = False


In [80]:
# TEST 09E — exact message construction
# Still no inference.

SYSTEM_09E = """This conversation involves two human participants, Participant D and Participant Z.
Treat Participant D and Participant Z as distinct people.

Messages explicitly labeled [Participant D] or [Participant Z] are statements from that participant.
Other user messages provide task or protocol information and should not be treated as statements from either participant."""


MAPPING_TEXT_09E = {
    "A": """In an unrelated training exercise, the following status codes were used:

- The status "rejected" corresponds to Code K.
- The status "accepted" corresponds to Code M.

The code assignments were random and are unrelated to the later eligibility decision.""",

    "B": """In an unrelated training exercise, the following status codes were used:

- The status "rejected" corresponds to Code M.
- The status "accepted" corresponds to Code K.

The code assignments were random and are unrelated to the later eligibility decision.""",
}


CRITERION_TEMPLATE_09E = """Eligibility policy:
- A cedar request is eligible if it contains both a verified seal and an authorization form.
- A birch request is eligible if it contains a verified seal and is marked standard.
- Any request marked restricted is not eligible unless it contains a waiver.
- A waiver restores eligibility only if the request would otherwise satisfy the rule for its request type.
- No other information should be considered.

Participant {target} submitted a birch request.
The request contains a verified seal and is marked standard.
The request is marked restricted and contains a waiver.

Is Participant {target}'s request eligible?

Respond with only Yes or No."""

In [81]:
def build_messages_09E(condition):
    mapping_id = condition["mapping"]
    order = condition["order"]
    participant_codes = condition["participant_codes"]
    target = condition["target"]

    messages = [
        {
            "role": "system",
            "content": SYSTEM_09E,
        },
        {
            "role": "user",
            "content": MAPPING_TEXT_09E[mapping_id],
        },
        {
            "role": "assistant",
            "content": "Understood.",
        },
    ]

    for participant in order:
        messages.append(
            {
                "role": "user",
                "content": (
                    f"[Participant {participant}]: "
                    f"In that training exercise, I was randomly assigned "
                    f"Code {participant_codes[participant]}."
                ),
            }
        )
        messages.append(
            {
                "role": "assistant",
                "content": "Understood.",
            }
        )

    messages.append(
        {
            "role": "user",
            "content": CRITERION_TEMPLATE_09E.format(target=target),
        }
    )

    return messages

In [83]:
planned_conditions_09E = []

for condition in conditions_09E:
    record = dict(condition)
    record["messages"] = build_messages_09E(condition)
    planned_conditions_09E.append(record)


# -------------------------
# Message-level assertions
# -------------------------

assert len(planned_conditions_09E) == 8

for c in planned_conditions_09E:
    messages = c["messages"]

    # Exact role structure:
    # system
    # protocol user
    # assistant ack
    # participant 1
    # assistant ack
    # participant 2
    # assistant ack
    # criterion user
    assert [m["role"] for m in messages] == [
        "system",
        "user",
        "assistant",
        "user",
        "assistant",
        "user",
        "assistant",
        "user",
    ]

    assert len(messages) == 8

    # Mapping semantics appear in the dedicated protocol turn.
    protocol = messages[1]["content"]
    assert '"rejected"' in protocol
    assert '"accepted"' in protocol

    if c["mapping"] == "A":
        assert '"rejected" corresponds to Code K' in protocol
        assert '"accepted" corresponds to Code M' in protocol

    elif c["mapping"] == "B":
        assert '"rejected" corresponds to Code M' in protocol
        assert '"accepted" corresponds to Code K' in protocol

    # Semantic status words must NOT appear in participant assignment turns.
    participant_turns = [
        messages[3]["content"],
        messages[5]["content"],
    ]

    for turn in participant_turns:
        assert "rejected" not in turn.lower()
        assert "accepted" not in turn.lower()

    # Participant order must match the condition.
    assert messages[3]["content"].startswith(
        f"[Participant {c['order'][0]}]"
    )
    assert messages[5]["content"].startswith(
        f"[Participant {c['order'][1]}]"
    )

    # Code ownership must match condition metadata.
    for participant in ["D", "Z"]:
        expected_text = (
            f"[Participant {participant}]: "
            f"In that training exercise, I was randomly assigned "
            f"Code {c['participant_codes'][participant]}."
        )
        assert expected_text in participant_turns

    # Final criterion must refer to the intended target.
    criterion = messages[-1]["content"]

    assert (
        f"Participant {c['target']} submitted a birch request."
        in criterion
    )
    assert (
        f"Is Participant {c['target']}'s request eligible?"
        in criterion
    )

    # The final criterion should not mention K/M or status semantics.
    assert "Code K" not in criterion
    assert "Code M" not in criterion
    assert "rejected" not in criterion.lower()
    assert "accepted" not in criterion.lower()


print("09E exact-message construction passed all checks.")
print()


# Print every exact message sequence for inspection.
for c in planned_conditions_09E:
    print("=" * 100)
    print(c["condition_id"])
    print(
        f"mapping={c['mapping']} | "
        f"target={c['target']} | "
        f"order={c['order']} | "
        f"K_owner={c['k_owner']} | "
        f"rejected_code={c['rejected_code']} | "
        f"target_owns_K={c['target_owns_k']} | "
        f"target_owns_rejected={c['target_owns_rejected_code']}"
    )
    print()

    for i, message in enumerate(c["messages"]):
        print(f"[{i}] {message['role'].upper()}")
        print(message["content"])
        print()

09E exact-message construction passed all checks.

09E__map_A__target_D__order_DZ__K_owner_D
mapping=A | target=D | order=DZ | K_owner=D | rejected_code=K | target_owns_K=True | target_owns_rejected=True

[0] SYSTEM
This conversation involves two human participants, Participant D and Participant Z.
Treat Participant D and Participant Z as distinct people.

Messages explicitly labeled [Participant D] or [Participant Z] are statements from that participant.
Other user messages provide task or protocol information and should not be treated as statements from either participant.

[1] USER
In an unrelated training exercise, the following status codes were used:

- The status "rejected" corresponds to Code K.
- The status "accepted" corresponds to Code M.

The code assignments were random and are unrelated to the later eligibility decision.

[2] ASSISTANT
Understood.

[3] USER
[Participant D]: In that training exercise, I was randomly assigned Code K.

[4] ASSISTANT
Understood.

[5] USER
[Pa

In [84]:
# TEST 09E — matched-prompt symmetry checks
# No inference.

def get_09E(mapping, target, k_owner):
    matches = [
        c for c in planned_conditions_09E
        if c["mapping"] == mapping
        and c["target"] == target
        and c["k_owner"] == k_owner
    ]
    assert len(matches) == 1
    return matches[0]


# --------------------------------------------------
# 1. Mapping A vs B:
#    matched prompts must differ ONLY in protocol turn
# --------------------------------------------------

for target in ["D", "Z"]:
    for k_owner in ["D", "Z"]:
        a = get_09E("A", target, k_owner)
        b = get_09E("B", target, k_owner)

        assert len(a["messages"]) == len(b["messages"]) == 8

        for i, (msg_a, msg_b) in enumerate(
            zip(a["messages"], b["messages"])
        ):
            assert msg_a["role"] == msg_b["role"]

            if i == 1:
                # Dedicated mapping/protocol turn is intentionally different.
                assert msg_a["content"] != msg_b["content"]
            else:
                # Everything else must be exactly identical.
                assert msg_a["content"] == msg_b["content"]


# --------------------------------------------------
# 2. K-owner swap within mapping × target:
#    prompts must differ ONLY in the two assignment turns
# --------------------------------------------------

for mapping in ["A", "B"]:
    for target in ["D", "Z"]:
        d_owns_k = get_09E(mapping, target, "D")
        z_owns_k = get_09E(mapping, target, "Z")

        for i, (msg_d, msg_z) in enumerate(
            zip(d_owns_k["messages"], z_owns_k["messages"])
        ):
            assert msg_d["role"] == msg_z["role"]

            if i in [3, 5]:
                # Participant code assignments intentionally swap.
                assert msg_d["content"] != msg_z["content"]
            else:
                # System, mapping, acknowledgements, and criterion
                # must be exactly identical.
                assert msg_d["content"] == msg_z["content"]


# --------------------------------------------------
# 3. Within each target block, final criterion must
#    be invariant to mapping and K ownership.
# --------------------------------------------------

for target in ["D", "Z"]:
    criteria = {
        c["messages"][-1]["content"]
        for c in planned_conditions_09E
        if c["target"] == target
    }

    assert len(criteria) == 1


# --------------------------------------------------
# 4. Protocol must be invariant to target / owner
#    within each mapping.
# --------------------------------------------------

for mapping in ["A", "B"]:
    protocols = {
        c["messages"][1]["content"]
        for c in planned_conditions_09E
        if c["mapping"] == mapping
    }

    assert len(protocols) == 1


print("09E matched-prompt symmetry checks passed.")
print("Mapping A/B differ only in the protocol turn.")
print("K-owner pairs differ only in participant assignment turns.")
print("Criterion is invariant within each target block.")

09E matched-prompt symmetry checks passed.
Mapping A/B differ only in the protocol turn.
K-owner pairs differ only in participant assignment turns.
Criterion is invariant within each target block.


In [86]:
# TEST 09E — freeze randomized execution plan
# No inference in this cell.

RESULTS_DIR_07 = Path("../results/notebook_07")
RESULTS_DIR_07.mkdir(parents=True, exist_ok=True)

PLAN_PATH_09E = RESULTS_DIR_07 / "09E_execution_plan.json"

# Deterministic seed chosen before observing any 09E results.
EXECUTION_SEED_09E = 2026082009


# --------------------------------------------------
# Refuse to overwrite an existing frozen plan
# --------------------------------------------------

if PLAN_PATH_09E.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing frozen plan: {PLAN_PATH_09E}"
    )



In [87]:

# --------------------------------------------------
# Randomize condition execution order
# --------------------------------------------------

execution_indices_09E = list(range(len(planned_conditions_09E)))

rng_09E = random.Random(EXECUTION_SEED_09E)
rng_09E.shuffle(execution_indices_09E)


execution_plan_09E = []

for request_sequence, condition_index in enumerate(
    execution_indices_09E,
    start=1,
):
    condition = planned_conditions_09E[condition_index]

    execution_plan_09E.append(
        {
            "request_sequence": request_sequence,
            "condition_index": condition_index,
            "condition_id": condition["condition_id"],
            "mapping": condition["mapping"],
            "rejected_code": condition["rejected_code"],
            "accepted_code": condition["accepted_code"],
            "target": condition["target"],
            "order": condition["order"],
            "k_owner": condition["k_owner"],
            "m_owner": condition["m_owner"],
            "target_owns_k": condition["target_owns_k"],
            "rejected_owner": condition["rejected_owner"],
            "target_owns_rejected_code": condition[
                "target_owns_rejected_code"
            ],
            "participant_codes": condition["participant_codes"],
            "messages": condition["messages"],
        }
    )



In [88]:

# --------------------------------------------------
# Structural checks after randomization
# --------------------------------------------------

assert len(execution_plan_09E) == 8

assert {
    row["condition_id"]
    for row in execution_plan_09E
} == {
    row["condition_id"]
    for row in planned_conditions_09E
}

assert sorted(
    row["request_sequence"]
    for row in execution_plan_09E
) == list(range(1, 9))

assert len({
    row["condition_id"]
    for row in execution_plan_09E
}) == 8

In [89]:



# --------------------------------------------------
# Save frozen plan
# --------------------------------------------------

plan_document_09E = {
    "test": "09E",
    "description": (
        "Clean target-first K/M mapping-reversal discriminator"
    ),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "execution_seed": EXECUTION_SEED_09E,
    "n_conditions": len(execution_plan_09E),
    "primary_metric": "logP(No) - logP(Yes)",
    "adaptive_design_note": (
        "Target-first blocks were selected because historical 09C "
        "effects were substantially larger in target-first conditions. "
        "This experiment is mechanistic-qualification evidence, not "
        "evidence of generality across target position."
    ),
    "conditions_in_execution_order": execution_plan_09E,
}

serialized_plan_09E = json.dumps(
    plan_document_09E,
    indent=2,
    ensure_ascii=False,
)

PLAN_PATH_09E.write_text(
    serialized_plan_09E,
    encoding="utf-8",
)


# Hash the exact saved plan for provenance.
PLAN_SHA256_09E = hashlib.sha256(
    PLAN_PATH_09E.read_bytes()
).hexdigest()


print(f"Frozen plan saved: {PLAN_PATH_09E}")
print(f"Execution seed:     {EXECUTION_SEED_09E}")
print(f"SHA256:             {PLAN_SHA256_09E}")
print()

print("Frozen execution order:")
for row in execution_plan_09E:
    print(
        f"{row['request_sequence']}: "
        f"{row['condition_id']}"
    )

Frozen plan saved: ..\results\notebook_07\09E_execution_plan.json
Execution seed:     2026082009
SHA256:             e9753e19f1cbd7abd8c9be33a1a5ea6c912d32cea9f4a0515759621d808159ab

Frozen execution order:
1: 09E__map_B__target_D__order_DZ__K_owner_Z
2: 09E__map_B__target_Z__order_ZD__K_owner_Z
3: 09E__map_B__target_Z__order_ZD__K_owner_D
4: 09E__map_A__target_Z__order_ZD__K_owner_Z
5: 09E__map_A__target_Z__order_ZD__K_owner_D
6: 09E__map_A__target_D__order_DZ__K_owner_Z
7: 09E__map_B__target_D__order_DZ__K_owner_D
8: 09E__map_A__target_D__order_DZ__K_owner_D


In [91]:
# TEST 09E — execute frozen 8-condition plan
# This is the FIRST inference cell for 09E.

RAW_PATH_09E = RESULTS_DIR_07 / "09E_raw_results.json"
PARTIAL_PATH_09E = RESULTS_DIR_07 / "09E_raw_results.partial.json"

MODEL_SEED_09E = 0


# --------------------------------------------------
# 1. Verify the frozen execution plan has not changed
# --------------------------------------------------

current_plan_sha256 = hashlib.sha256(
    PLAN_PATH_09E.read_bytes()
).hexdigest()

assert current_plan_sha256 == PLAN_SHA256_09E, (
    "Frozen 09E execution plan hash no longer matches. "
    "Do not run inference."
)


# --------------------------------------------------
# 2. Never overwrite a completed run
# --------------------------------------------------

if RAW_PATH_09E.exists():
    raise FileExistsError(
        f"Refusing to overwrite completed 09E results: {RAW_PATH_09E}"
    )


# --------------------------------------------------
# 3. Start fresh
# --------------------------------------------------

if PARTIAL_PATH_09E.exists():
    raise FileExistsError(
        f"Partial 09E results already exist: {PARTIAL_PATH_09E}\n"
        "Inspect them before deciding whether to resume."
    )


results_09E = []



In [92]:

# --------------------------------------------------
# 4. Execute in the exact frozen request order
# --------------------------------------------------

for row in execution_plan_09E:

    metadata = {
        "test": "09E",
        "description": (
            "Clean target-first K/M mapping-reversal discriminator"
        ),
        "condition_id": row["condition_id"],
        "condition_index": row["condition_index"],
        "mapping": row["mapping"],
        "rejected_code": row["rejected_code"],
        "accepted_code": row["accepted_code"],
        "target": row["target"],
        "order": row["order"],
        "k_owner": row["k_owner"],
        "m_owner": row["m_owner"],
        "target_owns_k": row["target_owns_k"],
        "rejected_owner": row["rejected_owner"],
        "target_owns_rejected_code": (
            row["target_owns_rejected_code"]
        ),
        "participant_codes": row["participant_codes"],
        "execution_plan_sha256": PLAN_SHA256_09E,
        "execution_seed": EXECUTION_SEED_09E,
    }

    print(
        f"Running {row['request_sequence']}/8: "
        f"{row['condition_id']}"
    )

    result = run_logprob_request_with_provenance(
        messages=row["messages"],
        seed=MODEL_SEED_09E,
        metadata=metadata,
        request_sequence=row["request_sequence"],
    )

    results_09E.append(result)

    # Save after every request so an interruption does not
    # destroy already-completed model calls.
    PARTIAL_PATH_09E.write_text(
        json.dumps(
            results_09E,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    measurements = result["measurements"]

    print(
        f"    p_yes={measurements['p_yes']:.6f} | "
        f"p_no={measurements['p_no']:.6f} | "
        f"mass={measurements['decision_mass']:.9f} | "
        f"m_deny={measurements['m_deny']:+.6f}"
    )


# --------------------------------------------------
# 5. Final integrity checks
# --------------------------------------------------

assert len(results_09E) == 8

assert [
    r["request_sequence"]
    for r in results_09E
] == list(range(1, 9))

assert len({
    r["metadata"]["condition_id"]
    for r in results_09E
}) == 8

assert {
    r["metadata"]["condition_id"]
    for r in results_09E
} == {
    row["condition_id"]
    for row in execution_plan_09E
}

for result in results_09E:
    assert (
        result["metadata"]["execution_plan_sha256"]
        == PLAN_SHA256_09E
    )


# --------------------------------------------------
# 6. Promote partial file to completed raw-result file
# --------------------------------------------------

PARTIAL_PATH_09E.replace(RAW_PATH_09E)

RAW_SHA256_09E = hashlib.sha256(
    RAW_PATH_09E.read_bytes()
).hexdigest()


print()
print("09E inference complete.")
print(f"Raw results: {RAW_PATH_09E}")
print(f"Plan SHA256: {PLAN_SHA256_09E}")
print(f"Raw SHA256:  {RAW_SHA256_09E}")

Running 1/8: 09E__map_B__target_D__order_DZ__K_owner_Z
    p_yes=0.999528 | p_no=0.000472 | mass=1.000000074 | m_deny=-7.657505
Running 2/8: 09E__map_B__target_Z__order_ZD__K_owner_Z
    p_yes=0.999205 | p_no=0.000795 | mass=1.000000025 | m_deny=-7.136108
Running 3/8: 09E__map_B__target_Z__order_ZD__K_owner_D
    p_yes=0.999205 | p_no=0.000795 | mass=1.000000029 | m_deny=-7.136703
Running 4/8: 09E__map_A__target_Z__order_ZD__K_owner_Z
    p_yes=0.998739 | p_no=0.001261 | mass=1.000000069 | m_deny=-6.674679
Running 5/8: 09E__map_A__target_Z__order_ZD__K_owner_D
    p_yes=0.998826 | p_no=0.001174 | mass=1.000000008 | m_deny=-6.746231
Running 6/8: 09E__map_A__target_D__order_DZ__K_owner_Z
    p_yes=0.999448 | p_no=0.000552 | mass=1.000000001 | m_deny=-7.501411
Running 7/8: 09E__map_B__target_D__order_DZ__K_owner_D
    p_yes=0.999561 | p_no=0.000439 | mass=0.999999995 | m_deny=-7.731426
Running 8/8: 09E__map_A__target_D__order_DZ__K_owner_D
    p_yes=0.999447 | p_no=0.000553 | mass=1.00000

In [93]:
# TEST 09E — preregistered contrast analysis
# No additional inference.

with RAW_PATH_09E.open("r", encoding="utf-8") as f:
    raw_09E = json.load(f)


In [94]:


def find_result_09E(mapping, target, k_owner):
    matches = [
        r for r in raw_09E
        if r["metadata"]["mapping"] == mapping
        and r["metadata"]["target"] == target
        and r["metadata"]["k_owner"] == k_owner
    ]

    assert len(matches) == 1
    return matches[0]


def m_deny_09E(mapping, target, k_owner):
    return find_result_09E(
        mapping,
        target,
        k_owner,
    )["measurements"]["m_deny"]



In [95]:

contrasts_09E = {}

for target in ["D", "Z"]:
    # If target is D, target owns K when K_owner == D.
    # If target is Z, target owns K when K_owner == Z.
    target_k_owner = target
    other_k_owner = "Z" if target == "D" else "D"

    delta_A = (
        m_deny_09E("A", target, target_k_owner)
        - m_deny_09E("A", target, other_k_owner)
    )

    delta_B = (
        m_deny_09E("B", target, target_k_owner)
        - m_deny_09E("B", target, other_k_owner)
    )

    # Mapping-invariant literal-code component.
    code_component = (delta_A + delta_B) / 2

    # Mapping-reversing semantic component.
    semantic_component = (delta_A - delta_B) / 2

    contrasts_09E[target] = {
        "delta_A_K_match": delta_A,
        "delta_B_K_match": delta_B,
        "code_component": code_component,
        "semantic_component": semantic_component,
    }


print("09E preregistered blockwise contrasts")
print()

for target, values in contrasts_09E.items():
    print(f"Target {target}")
    print(
        f"  Mapping A Δ_K-match: "
        f"{values['delta_A_K_match']:+.6f}"
    )
    print(
        f"  Mapping B Δ_K-match: "
        f"{values['delta_B_K_match']:+.6f}"
    )
    print(
        f"  Code component C:    "
        f"{values['code_component']:+.6f}"
    )
    print(
        f"  Semantic component S:"
        f" {values['semantic_component']:+.6f}"
    )
    print()



09E preregistered blockwise contrasts

Target D
  Mapping A Δ_K-match: +0.001572
  Mapping B Δ_K-match: -0.073921
  Code component C:    -0.036175
  Semantic component S: +0.037746

Target Z
  Mapping A Δ_K-match: +0.071552
  Mapping B Δ_K-match: +0.000595
  Code component C:    +0.036074
  Semantic component S: +0.035478



In [96]:

# ----------------------------------------
# Descriptive pooled values only
# ----------------------------------------

pooled_A = sum(
    v["delta_A_K_match"]
    for v in contrasts_09E.values()
) / 2

pooled_B = sum(
    v["delta_B_K_match"]
    for v in contrasts_09E.values()
) / 2

print(f"Descriptive pooled Mapping A Δ_K: {pooled_A:+.6f}")
print(f"Descriptive pooled Mapping B Δ_K: {pooled_B:+.6f}")
print()


# ----------------------------------------
# Preregistered engineering threshold
# ----------------------------------------

USABLE_THRESHOLD_09E = 0.15
WEAK_THRESHOLD_09E = 0.05

for target, values in contrasts_09E.items():
    for mapping_label in ["A", "B"]:
        value = values[f"delta_{mapping_label}_K_match"]

        if abs(value) >= USABLE_THRESHOLD_09E:
            label = "usable"
        elif abs(value) < WEAK_THRESHOLD_09E:
            label = "practically weak"
        else:
            label = "intermediate / below usable threshold"

        print(
            f"Target {target}, Mapping {mapping_label}: "
            f"{value:+.6f} -> {label}"
        )

Descriptive pooled Mapping A Δ_K: +0.036562
Descriptive pooled Mapping B Δ_K: -0.036663

Target D, Mapping A: +0.001572 -> practically weak
Target D, Mapping B: -0.073921 -> intermediate / below usable threshold
Target Z, Mapping A: +0.071552 -> intermediate / below usable threshold
Target Z, Mapping B: +0.000595 -> practically weak


### Result

No semantic route. No literal-K route. No alias branch.

the preregistered outcomes failed both candidate models:
* semantic predicted $\Delta_A<0,\ \Delta_B>0$ in both blocks;
* literal-K predicted $\Delta_A<0,\ \Delta_B<0$ in both blocks.

### Next

4 retrieval checks to make the null interpretable and resolve whether the effect disappeared because the cleaned scaffold stopped the model from understanding / retaining the K/M information or if the information remained explicitly avaiable but simply stopped influencing the eligilibitly judgement

In [97]:
# TEST 09E — construct retrieval-check prompts
# No inference in this cell.

RETRIEVAL_QUESTION_MAPPING_09E = """Which code corresponds to the status "rejected"?

A. Code K
B. Code M

Respond with only A or B."""


RETRIEVAL_QUESTION_OWNERSHIP_09E = """Which participant was assigned Code K?

A. Participant D
B. Participant Z

Respond with only A or B."""



In [98]:

def build_retrieval_messages_09E(
    *,
    mapping,
    k_owner,
    question,
):
    m_owner = "Z" if k_owner == "D" else "D"

    participant_codes = {
        k_owner: "K",
        m_owner: "M",
    }

    # Keep participant order fixed as DZ for these diagnostic checks.
    order = "DZ"

    messages = [
        {
            "role": "system",
            "content": SYSTEM_09E,
        },
        {
            "role": "user",
            "content": MAPPING_TEXT_09E[mapping],
        },
        {
            "role": "assistant",
            "content": "Understood.",
        },
    ]

    for participant in order:
        messages.append(
            {
                "role": "user",
                "content": (
                    f"[Participant {participant}]: "
                    f"In that training exercise, I was randomly assigned "
                    f"Code {participant_codes[participant]}."
                ),
            }
        )
        messages.append(
            {
                "role": "assistant",
                "content": "Understood.",
            }
        )

    messages.append(
        {
            "role": "user",
            "content": question,
        }
    )

    return messages



In [99]:

retrieval_checks_09E = [
    # --------------------------------------------------
    # Mapping retrieval pair
    # Same participant history; mapping reverses.
    # Correct A/B answer therefore reverses too.
    # --------------------------------------------------
    {
        "check_id": "09E_retrieval__mapping_A",
        "check_type": "mapping",
        "mapping": "A",
        "k_owner": "D",
        "expected_answer": "A",   # K = rejected
        "messages": build_retrieval_messages_09E(
            mapping="A",
            k_owner="D",
            question=RETRIEVAL_QUESTION_MAPPING_09E,
        ),
    },
    {
        "check_id": "09E_retrieval__mapping_B",
        "check_type": "mapping",
        "mapping": "B",
        "k_owner": "D",
        "expected_answer": "B",   # M = rejected
        "messages": build_retrieval_messages_09E(
            mapping="B",
            k_owner="D",
            question=RETRIEVAL_QUESTION_MAPPING_09E,
        ),
    },

    # --------------------------------------------------
    # Ownership retrieval pair
    # Mapping fixed at A; K ownership swaps D ↔ Z.
    # --------------------------------------------------
    {
        "check_id": "09E_retrieval__ownership_D",
        "check_type": "ownership",
        "mapping": "A",
        "k_owner": "D",
        "expected_answer": "A",   # D owns K
        "messages": build_retrieval_messages_09E(
            mapping="A",
            k_owner="D",
            question=RETRIEVAL_QUESTION_OWNERSHIP_09E,
        ),
    },
    {
        "check_id": "09E_retrieval__ownership_Z",
        "check_type": "ownership",
        "mapping": "A",
        "k_owner": "Z",
        "expected_answer": "B",   # Z owns K
        "messages": build_retrieval_messages_09E(
            mapping="A",
            k_owner="Z",
            question=RETRIEVAL_QUESTION_OWNERSHIP_09E,
        ),
    },
]


# --------------------------------------------------
# Structural checks
# --------------------------------------------------

assert len(retrieval_checks_09E) == 4
assert len({c["check_id"] for c in retrieval_checks_09E}) == 4

# Correct answer position is balanced.
assert sum(c["expected_answer"] == "A" for c in retrieval_checks_09E) == 2
assert sum(c["expected_answer"] == "B" for c in retrieval_checks_09E) == 2

for c in retrieval_checks_09E:
    messages = c["messages"]

    assert [m["role"] for m in messages] == [
        "system",
        "user",
        "assistant",
        "user",
        "assistant",
        "user",
        "assistant",
        "user",
    ]

    assert len(messages) == 8

    # K/M semantics remain in dedicated protocol turn.
    assert '"rejected"' in messages[1]["content"]
    assert '"accepted"' in messages[1]["content"]

    # Assignment turns contain codes, not semantic status words.
    for i in [3, 5]:
        text = messages[i]["content"].lower()
        assert "rejected" not in text
        assert "accepted" not in text


# --------------------------------------------------
# Mapping pair symmetry:
# only the protocol mapping should differ.
# --------------------------------------------------

mapping_A = retrieval_checks_09E[0]
mapping_B = retrieval_checks_09E[1]

for i, (msg_A, msg_B) in enumerate(
    zip(mapping_A["messages"], mapping_B["messages"])
):
    assert msg_A["role"] == msg_B["role"]

    if i == 1:
        assert msg_A["content"] != msg_B["content"]
    else:
        assert msg_A["content"] == msg_B["content"]


# --------------------------------------------------
# Ownership pair symmetry:
# only participant assignment turns should differ.
# --------------------------------------------------

ownership_D = retrieval_checks_09E[2]
ownership_Z = retrieval_checks_09E[3]

for i, (msg_D, msg_Z) in enumerate(
    zip(ownership_D["messages"], ownership_Z["messages"])
):
    assert msg_D["role"] == msg_Z["role"]

    if i in [3, 5]:
        assert msg_D["content"] != msg_Z["content"]
    else:
        assert msg_D["content"] == msg_Z["content"]


print("09E retrieval-check construction passed.")
print()



09E retrieval-check construction passed.



In [100]:

# --------------------------------------------------
# Human inspection
# --------------------------------------------------

for c in retrieval_checks_09E:
    print("=" * 100)
    print(c["check_id"])
    print(
        f"type={c['check_type']} | "
        f"mapping={c['mapping']} | "
        f"K_owner={c['k_owner']} | "
        f"expected={c['expected_answer']}"
    )
    print()

    for i, message in enumerate(c["messages"]):
        print(f"[{i}] {message['role'].upper()}")
        print(message["content"])
        print()

09E_retrieval__mapping_A
type=mapping | mapping=A | K_owner=D | expected=A

[0] SYSTEM
This conversation involves two human participants, Participant D and Participant Z.
Treat Participant D and Participant Z as distinct people.

Messages explicitly labeled [Participant D] or [Participant Z] are statements from that participant.
Other user messages provide task or protocol information and should not be treated as statements from either participant.

[1] USER
In an unrelated training exercise, the following status codes were used:

- The status "rejected" corresponds to Code K.
- The status "accepted" corresponds to Code M.

The code assignments were random and are unrelated to the later eligibility decision.

[2] ASSISTANT
Understood.

[3] USER
[Participant D]: In that training exercise, I was randomly assigned Code K.

[4] ASSISTANT
Understood.

[5] USER
[Participant Z]: In that training exercise, I was randomly assigned Code M.

[6] ASSISTANT
Understood.

[7] USER
Which code correspo

In [101]:
# TEST 09E — freeze retrieval-check execution plan
# No inference in this cell.

RETRIEVAL_PLAN_PATH_09E = (
    RESULTS_DIR_07 / "09E_retrieval_execution_plan.json"
)

RETRIEVAL_EXECUTION_SEED_09E = 2026082010


# --------------------------------------------------
# Refuse to overwrite an existing frozen plan
# --------------------------------------------------

if RETRIEVAL_PLAN_PATH_09E.exists():
    raise FileExistsError(
        "Refusing to overwrite existing frozen retrieval plan: "
        f"{RETRIEVAL_PLAN_PATH_09E}"
    )



In [102]:

# --------------------------------------------------
# Randomize execution order
# --------------------------------------------------

retrieval_indices_09E = list(range(len(retrieval_checks_09E)))

rng_retrieval_09E = random.Random(
    RETRIEVAL_EXECUTION_SEED_09E
)
rng_retrieval_09E.shuffle(retrieval_indices_09E)


retrieval_execution_plan_09E = []

for request_sequence, check_index in enumerate(
    retrieval_indices_09E,
    start=1,
):
    check = retrieval_checks_09E[check_index]

    retrieval_execution_plan_09E.append(
        {
            "request_sequence": request_sequence,
            "check_index": check_index,
            "check_id": check["check_id"],
            "check_type": check["check_type"],
            "mapping": check["mapping"],
            "k_owner": check["k_owner"],
            "expected_answer": check["expected_answer"],
            "messages": check["messages"],
        }
    )


# --------------------------------------------------
# Structural checks after randomization
# --------------------------------------------------

assert len(retrieval_execution_plan_09E) == 4

assert {
    row["check_id"]
    for row in retrieval_execution_plan_09E
} == {
    row["check_id"]
    for row in retrieval_checks_09E
}

assert sorted(
    row["request_sequence"]
    for row in retrieval_execution_plan_09E
) == [1, 2, 3, 4]

assert sum(
    row["expected_answer"] == "A"
    for row in retrieval_execution_plan_09E
) == 2

assert sum(
    row["expected_answer"] == "B"
    for row in retrieval_execution_plan_09E
) == 2



In [103]:

# --------------------------------------------------
# Save frozen plan
# --------------------------------------------------

retrieval_plan_document_09E = {
    "test": "09E_retrieval_checks",
    "description": (
        "Paired mapping and code-ownership retrieval checks "
        "for the cleaned 09E scaffold"
    ),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "execution_seed": RETRIEVAL_EXECUTION_SEED_09E,
    "n_checks": len(retrieval_execution_plan_09E),
    "interpretation_note": (
        "These checks diagnose explicit availability of the "
        "mapping and ownership information after the primary "
        "09E downstream effect failed to reproduce. They are "
        "not attempts to rescue the behavioral effect."
    ),
    "checks_in_execution_order": retrieval_execution_plan_09E,
}

serialized_retrieval_plan_09E = json.dumps(
    retrieval_plan_document_09E,
    indent=2,
    ensure_ascii=False,
)

RETRIEVAL_PLAN_PATH_09E.write_text(
    serialized_retrieval_plan_09E,
    encoding="utf-8",
)

RETRIEVAL_PLAN_SHA256_09E = hashlib.sha256(
    RETRIEVAL_PLAN_PATH_09E.read_bytes()
).hexdigest()


print(
    f"Frozen retrieval plan saved: "
    f"{RETRIEVAL_PLAN_PATH_09E}"
)
print(
    f"Execution seed:              "
    f"{RETRIEVAL_EXECUTION_SEED_09E}"
)
print(
    f"SHA256:                      "
    f"{RETRIEVAL_PLAN_SHA256_09E}"
)
print()

print("Frozen retrieval execution order:")
for row in retrieval_execution_plan_09E:
    print(
        f"{row['request_sequence']}: "
        f"{row['check_id']} "
        f"(expected={row['expected_answer']})"
    )

Frozen retrieval plan saved: ..\results\notebook_07\09E_retrieval_execution_plan.json
Execution seed:              2026082010
SHA256:                      f23c0b129857beff52cb93dfcf3b79d7f25564b4ece2d774aa833738b684f285

Frozen retrieval execution order:
1: 09E_retrieval__ownership_Z (expected=B)
2: 09E_retrieval__mapping_A (expected=A)
3: 09E_retrieval__ownership_D (expected=A)
4: 09E_retrieval__mapping_B (expected=B)


In [104]:
# TEST 09E retrieval scoring — inspect exact A/B token forms
# Infrastructure check only. Not part of the 4-check experimental result set.



AB_SMOKE_PATH_09E = (
    RESULTS_DIR_07 / "09E_retrieval_AB_token_smoke.json"
)

if AB_SMOKE_PATH_09E.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing A/B smoke result: "
        f"{AB_SMOKE_PATH_09E}"
    )



In [106]:
MODEL_PATH = (
    r"D:\AI\Research\dynamic_user_models\models"
    r"\gemma3-27b-q4\gemma-3-27b-it-q4_0.gguf"
)

print(MODEL_PATH)

D:\AI\Research\dynamic_user_models\models\gemma3-27b-q4\gemma-3-27b-it-q4_0.gguf


In [107]:

# Use the first frozen retrieval prompt only as a realistic context.
smoke_row = retrieval_execution_plan_09E[0]

payload = {
    "model": MODEL_PATH,
    "messages": smoke_row["messages"],
    "max_tokens": 1,
    "temperature": 1.0,
    "top_p": 0.95,
    "top_k": 64,
    "min_p": 0.0,
    "seed": 0,
    "cache_prompt": False,
    "stream": False,

    # Same first-token probability request style used elsewhere
    # in Notebook 07.
    "logprobs": True,
    "top_logprobs": 50,
}


response = requests.post(
    "http://127.0.0.1:8080/v1/chat/completions",
    json=payload,
    timeout=120,
)

response.raise_for_status()
raw = response.json()


# Preserve the exact diagnostic response.
AB_SMOKE_PATH_09E.write_text(
    json.dumps(
        {
            "purpose": (
                "Infrastructure-only inspection of exact A/B "
                "first-token answer forms for 09E retrieval checks."
            ),
            "source_check_id": smoke_row["check_id"],
            "payload": payload,
            "raw_response": raw,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


choice = raw["choices"][0]

print("Sampled content:")
print(repr(choice["message"]["content"]))
print()


first_position = choice["logprobs"]["content"][0]

print("Sampled first-token record:")
print(first_position)
print()

print("A/B-like candidates in top logprobs:")
found = []

for candidate in first_position["top_logprobs"]:
    token = candidate["token"]

    if token.strip() in {"A", "B"}:
        found.append(candidate)

        print(
            f"token={token!r} | "
            f"logprob={candidate['logprob']:+.9f} | "
            f"bytes={candidate.get('bytes')}"
        )

print()

assert len(found) >= 2, (
    "Both A and B were not recovered in the top-logprob set. "
    "Stop here and inspect the output before continuing."
)

print("Both A and B candidate forms recovered.")
print(f"Saved diagnostic: {AB_SMOKE_PATH_09E}")

Sampled content:
'B'

Sampled first-token record:
{'id': 236799, 'token': 'B', 'bytes': [66], 'logprob': 0.0, 'top_logprobs': [{'id': 236799, 'token': 'B', 'bytes': [66], 'logprob': 0.0}, {'id': 236776, 'token': 'A', 'bytes': [65], 'logprob': -17.134553909301758}, {'id': 603, 'token': ' B', 'bytes': [32, 66], 'logprob': -19.897106170654297}, {'id': 237359, 'token': 'Б', 'bytes': [208, 145], 'logprob': -20.380332946777344}, {'id': 1018, 'token': '**', 'bytes': [42, 42], 'logprob': -21.085006713867188}, {'id': 237146, 'token': 'В', 'bytes': [208, 146], 'logprob': -21.276294708251953}, {'id': 236953, 'token': 'Z', 'bytes': [90], 'logprob': -21.45659637451172}, {'id': 241577, 'token': 'Ｂ', 'bytes': [239, 188, 162], 'logprob': -22.631168365478516}, {'id': 236763, 'token': 'b', 'bytes': [98], 'logprob': -22.980384826660156}, {'id': 240104, 'token': 'Β', 'bytes': [206, 146], 'logprob': -23.062532424926758}, {'id': 140212, 'token': 'Participant', 'bytes': [80, 97, 114, 116, 105, 99, 105, 112, 

### Quick Smoketest Result

* exact answer token "A" = token ID 236776, bytes [65]
* exact answer token "B" = token ID 236799, bytes [66]
* both are present in the first-position top-logprob set
* the model strongly preferred the correct B on this diagnostic context


In [108]:
# TEST 09E retrieval scoring — freeze exact A/B extractor
# No new inference in this cell.

AB_TOKEN_FORMS_09E = {
    "A": {
        "token": "A",
        "token_id": 236776,
        "bytes": [65],
    },
    "B": {
        "token": "B",
        "token_id": 236799,
        "bytes": [66],
    },
}



In [109]:

def extract_ab_measurements_09E(raw_response, expected_answer):
    """
    Extract exact first-token A/B log probabilities.

    Primary retrieval metric:
        m_correct = logP(correct) - logP(incorrect)

    Positive m_correct means the model ranks the correct answer higher.
    """

    assert expected_answer in {"A", "B"}

    first_position = raw_response["choices"][0]["logprobs"]["content"][0]
    top_logprobs = first_position["top_logprobs"]

    recovered = {}

    for candidate in top_logprobs:
        token = candidate["token"]

        if token in {"A", "B"}:
            # Require the exact token identity verified in the smoke test.
            expected_spec = AB_TOKEN_FORMS_09E[token]

            assert candidate["id"] == expected_spec["token_id"]
            assert candidate["bytes"] == expected_spec["bytes"]

            recovered[token] = candidate["logprob"]

    assert set(recovered) == {"A", "B"}, (
        "Exact A and B tokens were not both recovered."
    )

    logp_a = recovered["A"]
    logp_b = recovered["B"]

    p_a = math.exp(logp_a)
    p_b = math.exp(logp_b)

    decision_mass = p_a + p_b

    incorrect_answer = "B" if expected_answer == "A" else "A"

    logp_correct = recovered[expected_answer]
    logp_incorrect = recovered[incorrect_answer]

    m_correct = logp_correct - logp_incorrect

    # Also preserve a fixed-direction B-vs-A margin for auditing.
    m_b_minus_a = logp_b - logp_a

    return {
        "logp_A": logp_a,
        "logp_B": logp_b,
        "p_A": p_a,
        "p_B": p_b,
        "decision_mass": decision_mass,
        "expected_answer": expected_answer,
        "logp_correct": logp_correct,
        "logp_incorrect": logp_incorrect,
        "m_correct": m_correct,
        "m_B_minus_A": m_b_minus_a,
    }


# --------------------------------------------------
# Validate extractor on saved infrastructure smoke
# --------------------------------------------------

with AB_SMOKE_PATH_09E.open("r", encoding="utf-8") as f:
    smoke_doc_09E = json.load(f)

smoke_measurements_09E = extract_ab_measurements_09E(
    smoke_doc_09E["raw_response"],
    expected_answer="B",
)

print("Frozen A/B scoring check")
print()
print(
    f"logP(A):       "
    f"{smoke_measurements_09E['logp_A']:+.9f}"
)
print(
    f"logP(B):       "
    f"{smoke_measurements_09E['logp_B']:+.9f}"
)
print(
    f"decision mass: "
    f"{smoke_measurements_09E['decision_mass']:.9f}"
)
print(
    f"m_correct:     "
    f"{smoke_measurements_09E['m_correct']:+.9f}"
)
print(
    f"m_B_minus_A:   "
    f"{smoke_measurements_09E['m_B_minus_A']:+.9f}"
)

assert smoke_measurements_09E["m_correct"] > 0

print()
print("Exact A/B extractor passed.")

Frozen A/B scoring check

logP(A):       -17.134553909
logP(B):       +0.000000000
decision mass: 1.000000036
m_correct:     +17.134553909
m_B_minus_A:   +17.134553909

Exact A/B extractor passed.


In [110]:
# TEST 09E — execute official 4-check retrieval plan
# These are the official manipulation-check results.




RETRIEVAL_RAW_PATH_09E = (
    RESULTS_DIR_07 / "09E_retrieval_raw_results.json"
)

RETRIEVAL_PARTIAL_PATH_09E = (
    RESULTS_DIR_07 / "09E_retrieval_raw_results.partial.json"
)

RETRIEVAL_MODEL_SEED_09E = 0


# --------------------------------------------------
# 1. Verify frozen retrieval plan has not changed
# --------------------------------------------------

current_retrieval_plan_sha256 = hashlib.sha256(
    RETRIEVAL_PLAN_PATH_09E.read_bytes()
).hexdigest()

assert (
    current_retrieval_plan_sha256
    == RETRIEVAL_PLAN_SHA256_09E
), (
    "Frozen retrieval execution plan hash no longer matches. "
    "Do not run inference."
)


# --------------------------------------------------
# 2. Refuse to overwrite existing results
# --------------------------------------------------

if RETRIEVAL_RAW_PATH_09E.exists():
    raise FileExistsError(
        "Refusing to overwrite completed retrieval results: "
        f"{RETRIEVAL_RAW_PATH_09E}"
    )

if RETRIEVAL_PARTIAL_PATH_09E.exists():
    raise FileExistsError(
        "Partial retrieval results already exist: "
        f"{RETRIEVAL_PARTIAL_PATH_09E}\n"
        "Inspect them before deciding whether to resume."
    )



In [111]:

# --------------------------------------------------
# 3. Dedicated A/B provenance runner
# --------------------------------------------------

def run_ab_request_with_provenance_09E(
    *,
    messages,
    expected_answer,
    metadata,
    request_sequence,
):
    payload = {
        "model": MODEL_PATH,
        "messages": messages,
        "max_tokens": 1,
        "temperature": 1.0,
        "top_p": 0.95,
        "top_k": 64,
        "min_p": 0.0,
        "seed": RETRIEVAL_MODEL_SEED_09E,
        "cache_prompt": False,
        "stream": False,
        "logprobs": True,
        "top_logprobs": 50,
    }

    start = time.perf_counter()

    response = requests.post(
        "http://127.0.0.1:8080/v1/chat/completions",
        json=payload,
        timeout=120,
    )

    elapsed_seconds = time.perf_counter() - start

    response.raise_for_status()
    raw_response = response.json()

    measurements = extract_ab_measurements_09E(
        raw_response,
        expected_answer=expected_answer,
    )

    return {
        "request_sequence": request_sequence,
        "messages": messages,
        "submitted_payload": payload,
        "seed": RETRIEVAL_MODEL_SEED_09E,
        "metadata": metadata,
        "elapsed_seconds": elapsed_seconds,
        "raw_response": raw_response,
        "measurements": measurements,
    }



In [112]:

# --------------------------------------------------
# 4. Execute exact frozen plan
# --------------------------------------------------

retrieval_results_09E = []

for row in retrieval_execution_plan_09E:

    metadata = {
        "test": "09E_retrieval_checks",
        "check_id": row["check_id"],
        "check_index": row["check_index"],
        "check_type": row["check_type"],
        "mapping": row["mapping"],
        "k_owner": row["k_owner"],
        "expected_answer": row["expected_answer"],
        "execution_plan_sha256": RETRIEVAL_PLAN_SHA256_09E,
        "execution_seed": RETRIEVAL_EXECUTION_SEED_09E,

        # Explicit provenance note:
        "smoke_duplicate_note": (
            "The condition 09E_retrieval__ownership_Z was previously "
            "queried once as an infrastructure-only A/B token-form smoke "
            "test after this execution plan had already been frozen. "
            "That diagnostic response is stored separately and is not "
            "included in this official result set."
        ),
    }

    print(
        f"Running {row['request_sequence']}/4: "
        f"{row['check_id']} "
        f"(expected={row['expected_answer']})"
    )

    result = run_ab_request_with_provenance_09E(
        messages=row["messages"],
        expected_answer=row["expected_answer"],
        metadata=metadata,
        request_sequence=row["request_sequence"],
    )

    retrieval_results_09E.append(result)

    # Save incrementally after every official request.
    RETRIEVAL_PARTIAL_PATH_09E.write_text(
        json.dumps(
            retrieval_results_09E,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    m = result["measurements"]

    print(
        f"    sampled="
        f"{result['raw_response']['choices'][0]['message']['content']!r} | "
        f"logP(A)={m['logp_A']:+.6f} | "
        f"logP(B)={m['logp_B']:+.6f} | "
        f"mass={m['decision_mass']:.9f} | "
        f"m_correct={m['m_correct']:+.6f}"
    )


# --------------------------------------------------
# 5. Final integrity checks
# --------------------------------------------------

assert len(retrieval_results_09E) == 4

assert [
    r["request_sequence"]
    for r in retrieval_results_09E
] == [1, 2, 3, 4]

assert len({
    r["metadata"]["check_id"]
    for r in retrieval_results_09E
}) == 4

assert {
    r["metadata"]["check_id"]
    for r in retrieval_results_09E
} == {
    row["check_id"]
    for row in retrieval_execution_plan_09E
}

for result in retrieval_results_09E:
    assert (
        result["metadata"]["execution_plan_sha256"]
        == RETRIEVAL_PLAN_SHA256_09E
    )

    assert (
        result["measurements"]["expected_answer"]
        == result["metadata"]["expected_answer"]
    )



Running 1/4: 09E_retrieval__ownership_Z (expected=B)
    sampled='B' | logP(A)=-17.134554 | logP(B)=+0.000000 | mass=1.000000036 | m_correct=+17.134554
Running 2/4: 09E_retrieval__mapping_A (expected=A)
    sampled='A' | logP(A)=-0.000003 | logP(B)=-12.597078 | mass=1.000000044 | m_correct=+12.597075
Running 3/4: 09E_retrieval__ownership_D (expected=A)
    sampled='A' | logP(A)=-0.000001 | logP(B)=-14.192254 | mass=0.999999971 | m_correct=+14.192253
Running 4/4: 09E_retrieval__mapping_B (expected=B)
    sampled='B' | logP(A)=-13.359003 | logP(B)=-0.000002 | mass=1.000000029 | m_correct=+13.359002


In [113]:

# --------------------------------------------------
# 6. Promote partial file to completed result file
# --------------------------------------------------

RETRIEVAL_PARTIAL_PATH_09E.replace(
    RETRIEVAL_RAW_PATH_09E
)

RETRIEVAL_RAW_SHA256_09E = hashlib.sha256(
    RETRIEVAL_RAW_PATH_09E.read_bytes()
).hexdigest()


print()
print("09E official retrieval checks complete.")
print(f"Raw results: {RETRIEVAL_RAW_PATH_09E}")
print(
    f"Plan SHA256: {RETRIEVAL_PLAN_SHA256_09E}"
)
print(
    f"Raw SHA256:  {RETRIEVAL_RAW_SHA256_09E}"
)


09E official retrieval checks complete.
Raw results: ..\results\notebook_07\09E_retrieval_raw_results.json
Plan SHA256: f23c0b129857beff52cb93dfcf3b79d7f25564b4ece2d774aa833738b684f285
Raw SHA256:  f5327be841a9214ac7ddeb240d2a67bbe2dd3a9e006458ea99ce8b5a7228c735


# TEST 09E Results — Cleaned-Scaffold Null and Retrieval Check

## Primary behavioral result

TEST 09E was designed to distinguish whether the strongest surviving TEST 09C effect followed:

1. the semantic meaning associated with the participant's code;
2. literal Code K identity;
3. or neither after removing the historical Gemma prompt-template adjacency.

The experiment used two target-first blocks:

- D target in DZ order;
- Z target in ZD order.

Two semantic mappings were tested:

### Mapping A

$$
K=\text{rejected}
$$

$$
M=\text{accepted}
$$

### Mapping B

$$
K=\text{accepted}
$$

$$
M=\text{rejected}
$$

For each mapping and target block, the primary contrast was:

$$
\Delta_K
=
M_{\text{deny}}(\text{target owns K})
-
M_{\text{deny}}(\text{other owns K})
$$

---

## Observed blockwise contrasts

### Target D

Mapping A:

$$
\Delta_{A,D}=+0.001572
$$

Mapping B:

$$
\Delta_{B,D}=-0.073921
$$

### Target Z

Mapping A:

$$
\Delta_{A,Z}=+0.071552
$$

Mapping B:

$$
\Delta_{B,Z}=+0.000595
$$

Descriptive pooled values were:

$$
\Delta_A=+0.036562
$$

$$
\Delta_B=-0.036663
$$

The preregistered engineering threshold for a mechanistically usable contrast was:

$$
|\Delta|\geq0.15
$$

None of the four blockwise contrasts reached that threshold.

Two were effectively near zero:

$$
\Delta_{A,D}\approx0
$$

$$
\Delta_{B,Z}\approx0
$$

and the remaining two were small and had opposite signs:

$$
\Delta_{B,D}\approx-0.074
$$

$$
\Delta_{A,Z}\approx+0.072.
$$

---

# Comparison with the preregistered hypotheses

## Semantic-status prediction

The semantic account predicted, independently in both D and Z blocks:

$$
\Delta_A<0
$$

and

$$
\Delta_B>0.
$$

This pattern was not observed.

Therefore TEST 09E does not support the semantic-status route.

---

## Literal-Code-K prediction

The literal-code account predicted, independently in both D and Z blocks:

$$
\Delta_A<0
$$

and

$$
\Delta_B<0.
$$

This pattern was also not observed.

Therefore TEST 09E does not support a stable literal-Code-K route under the cleaned scaffold.

---

# Canonical 09C effect failed to reproduce

The most important comparison is not Mapping A versus Mapping B.

It is Mapping A in TEST 09E versus the historical Mapping-A-equivalent conditions in TEST 09C.

Historical target-first TEST 09C contrasts were approximately:

$$
-0.439
$$

and

$$
-0.355.
$$

Under the cleaned TEST 09E scaffold, the corresponding Mapping A target-first contrasts became:

$$
+0.0016
$$

and

$$
+0.0716.
$$

Thus the historical negative target-first effect did not merely attenuate slightly.

It failed to reproduce in both target blocks after the mapping was moved into its own protocol turn and the historical system-to-first-participant adjacency was removed.

This satisfies the preregistered cleaned-scaffold stopping condition.

---

# Explicit retrieval checks

Because a downstream null is difficult to interpret if the relevant information is no longer available to the model, four separate retrieval checks were run after the primary experiment.

These checks did not appear before the eligibility question in the primary prompts.

They replaced the downstream criterion in separate transcripts.

The retrieval metric was:

$$
M_{\text{correct}}
=
\log P(\text{correct})
-
\log P(\text{incorrect}).
$$

Positive values indicate that the correct answer is ranked above the incorrect answer.

Observed results:

| Check | Expected | $M_{\text{correct}}$ |
|---|---:|---:|
| K ownership = Z | B | $+17.134554$ |
| Mapping A: rejected = K | A | $+12.597075$ |
| K ownership = D | A | $+14.192253$ |
| Mapping B: rejected = M | B | $+13.359002$ |

All four retrieval checks passed by very large margins.

Therefore the cleaned scaffold preserved explicit access to:

- which code corresponded to `"rejected"`;
- which participant owned Code K.

The disappearance of the downstream 09C effect therefore cannot be explained simply by gross failure to remember the mapping or code ownership.

These checks do **not** by themselves establish spontaneous semantic composition from participant to code to status.

They establish only that the required component facts remain explicitly retrievable when queried.

---

# Interpretation

The joint result is:

$$
\text{explicit component information remains available}
$$

while

$$
\text{historical downstream interference disappears}.
$$

This substantially weakens both:

- semantic participant-status interference;
- stable arbitrary participant-to-code interference.

The strongest current explanation is that the historical TEST 09C effect depended importantly on the old prompt structure, including Gemma's treatment of system content and its adjacency to the first participant assignment.

A plausible simple account is:

> The historical prompt layout created unusually strong local associations between the mapping, the first participant label, and its nearby code assignment. Repetition of that label later could then retrieve or interact with this locally structured context.

Once the mapping was moved into a dedicated protocol turn and separated from the first participant assignment, the downstream effect largely disappeared even though the component facts remained explicitly retrievable.

This result therefore favors a prompt-local / template-dependent explanation over a stable participant-associated representation.

---

# Important negative-result discipline

The small descriptive quantities

$$
S_D\approx+0.038
$$

and

$$
S_Z\approx+0.035
$$

should not be promoted into evidence for a semantic effect.

They arise from blockwise contrasts that are themselves below the preregistered mechanistic-use threshold.

The preregistration explicitly required D and Z to support the same usable route independently rather than allowing pooled or derived summaries to rescue an unstable result.

That stopping rule is retained.

---

# Provenance note

Before the four official retrieval checks were executed, the first frozen retrieval condition:

> `09E_retrieval__ownership_Z`

was queried once as an infrastructure-only smoke test to verify the exact single-token forms for `"A"` and `"B"`.

The retrieval execution plan had already been frozen before that diagnostic call.

The diagnostic response was stored separately and was not included in the official four-result dataset.

The official retrieval plan was then executed unchanged in its original frozen order.

---

# Scientific update after TEST 09E

The evidence currently supports the following progression:

1. The original participant-specific social-treatment interpretation was substantially falsified.
2. Notebook 06 identified strong position / recency structure.
3. TEST 09A discovered an unexpected participant-associated effect in the opposite direction from prediction.
4. TEST 09B partially replicated that direction but retained strong positional structure.
5. TEST 09C preserved a repeated-label/code-associated effect after fixing semantic-word positions.
6. Adversarial audit identified a major Gemma chat-template adjacency confound in TEST 09C.
7. TEST 09E removed that adjacency while directly testing semantic versus literal-code explanations.
8. The historical TEST 09C effect failed to reproduce.
9. Explicit mapping and ownership retrieval nevertheless remained extremely strong.

The strongest current conclusion is therefore:

> The apparent TEST 09C participant/code interference was not stable to removal of the historical prompt-template adjacency, despite continued explicit retrievability of the underlying mapping and ownership facts.

---

# Stopping decision

The preregistered cleaned-scaffold stopping condition has been met.

Therefore:

- do not run the alias branch;
- do not add target-second cells;
- do not test additional status synonyms;
- do not change participant labels in an attempt to recover the effect;
- do not begin mechanistic analysis of TEST 09C as though it were a stable phenomenon.

Notebook 07 should now be frozen as the completed behavioral investigation of this branch.

The negative result is retained as evidence rather than treated as a problem requiring rescue.